# Raízen S.A. — Análise Macro + Concorrentes + Red Flags
**Continuação do notebook principal**

Este notebook complementa a análise financeira e estatística com:
- **Seção 14:** Análise macro setorial — mercado de etanol
- **Seção 15:** Guerra de custos — milho vs cana
- **Seção 16:** Análise comparativa de concorrentes
- **Seção 17:** Red flags financeiros (DRE + DFC + Balanço)
- **Seção 18:** Red flags setoriais (macro + competitivos)
- **Seção 19:** Cronologia da deterioração
- **Seção 20:** Resumo final integrado

> **Regras:** Python puro · sem pandas · sem numpy · sem bibliotecas externas  
> **Dados:** CVM/DFP + fontes setoriais (BTG Pactual, Moody's, Datagro, UNEM)

---
## 14. Análise Macro Setorial — Mercado de Etanol no Brasil

Fonte dos dados: UNEM, Datagro, BTG Pactual, CONAB, ANP

In [1]:
import math

# ══════════════════════════════════════════════════════════════════════════════
# DADOS MACRO — Mercado de etanol no Brasil
# Fonte: UNEM, Datagro, BTG Pactual, CONAB
# ══════════════════════════════════════════════════════════════════════════════

# Produção nacional de etanol (bilhões de litros)
# Safras 2019/20 até 2024/25
SAFRAS = ['2019/20','2020/21','2021/22','2022/23','2023/24','2024/25','2025/26p']

prod_cana  = [29.8, 29.1, 29.3, 29.8, 30.2, 30.5, 28.0]  # bi litros
prod_milho = [ 2.0,  2.6,  4.0,  6.0,  7.8, 10.2, 10.5]  # bi litros
prod_total = [p + m for p, m in zip(prod_cana, prod_milho)]

# Market share do milho (%)
share_milho = [m / t * 100 for m, t in zip(prod_milho, prod_total)]

# Número de plantas de etanol de milho (operando)
plantas_op  = [5, 8, 12, 18, 25, 30, 47]   # última = projeção

# Projeção Datagro/BTG para 2030 e 2034
PROJ_2030 = {'milho': 18.0, 'cana': 27.0, 'share_milho': 40.0}
PROJ_2034 = {'milho': 24.7, 'cana': 26.0, 'share_milho': 49.0}

def media(a): return sum(a) / len(a)
def crescimento_cagr(inicio, fim, n_anos):
    """CAGR: taxa de crescimento anual composta."""
    return (fim / inicio) ** (1 / n_anos) - 1

print('=' * 65)
print('  ANÁLISE MACRO — MERCADO DE ETANOL BRASIL')
print('  Fonte: UNEM · Datagro · BTG Pactual · CONAB · ANP')
print('=' * 65)

# Tabela de produção
hdr = f"  {'Safra':<12}{'Cana (bi L)':>12}{'Milho (bi L)':>14}{'Total (bi L)':>14}{'Share milho':>13}"
print(hdr)
print('  ' + '─' * 63)
for i, s in enumerate(SAFRAS):
    flag = '  ← projeção' if 'p' in s else ''
    print(f"  {s:<12}{prod_cana[i]:>12.1f}{prod_milho[i]:>14.1f}"
          f"{prod_total[i]:>14.1f}{share_milho[i]:>12.1f}%{flag}")
print('  ' + '─' * 63)

# Estatísticas do crescimento
cagr_milho = crescimento_cagr(prod_milho[0], prod_milho[5], 5)
cagr_cana  = crescimento_cagr(prod_cana[0],  prod_cana[5],  5)
var_share  = share_milho[5] - share_milho[0]

print(f"""
  INDICADORES DE CRESCIMENTO:
  CAGR etanol de milho (2019/20 → 2024/25): {cagr_milho:.1%} ao ano
  CAGR etanol de cana  (2019/20 → 2024/25): {cagr_cana:.1%} ao ano
  Variação share do milho:                  +{var_share:.1f} p.p. em 5 anos

  PROJEÇÕES (Datagro / BTG Pactual / Itaú BBA — consenso de mercado):
  2030: milho = {PROJ_2030['milho']:.1f} bi L · share = {PROJ_2030['share_milho']:.0f}%
        (dobrando em relação ao share atual de ~25%)
  2034: milho = {PROJ_2034['milho']:.1f} bi L — igualando a produção de cana

  CONTEXTO DA RAÍZEN:
  A empresa apostou R$ 36+ bi em E2G (etanol do bagaço da cana)
  exatamente no período em que o milho crescia {cagr_milho:.0%}/ano.
  O mercado recompensou quem apostou em tecnologia simples e barata.
""")

# Análise univariada do share do milho
s = sorted(share_milho)
p25 = s[0] + (s[1]-s[0]) * 0.75
p75 = s[4] + (s[5]-s[4]) * 0.75
print('  UNIVARIADA — share do etanol de milho (%):')
print(f'  n={len(share_milho)} · Mín={min(share_milho):.1f} · Máx={max(share_milho):.1f}')
print(f'  Média={media(share_milho):.1f} · Amplitude={max(share_milho)-min(share_milho):.1f} p.p.')
print(f'  Crescimento médio: +{(share_milho[-1]-share_milho[0])/5:.1f} p.p./ano')

  ANÁLISE MACRO — MERCADO DE ETANOL BRASIL
  Fonte: UNEM · Datagro · BTG Pactual · CONAB · ANP
  Safra        Cana (bi L)  Milho (bi L)  Total (bi L)  Share milho
  ───────────────────────────────────────────────────────────────
  2019/20             29.8           2.0          31.8         6.3%
  2020/21             29.1           2.6          31.7         8.2%
  2021/22             29.3           4.0          33.3        12.0%
  2022/23             29.8           6.0          35.8        16.8%
  2023/24             30.2           7.8          38.0        20.5%
  2024/25             30.5          10.2          40.7        25.1%
  2025/26p            28.0          10.5          38.5        27.3%  ← projeção
  ───────────────────────────────────────────────────────────────

  INDICADORES DE CRESCIMENTO:
  CAGR etanol de milho (2019/20 → 2024/25): 38.5% ao ano
  CAGR etanol de cana  (2019/20 → 2024/25): 0.5% ao ano
  Variação share do milho:                  +18.8 p.p. em 5 anos

  PROJE

---
## 15. Guerra de Custos — Milho vs Cana vs E2G

Fonte: BTG Pactual (abr/2025) · Itaú BBA (set/2025)

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# CUSTOS DE PRODUÇÃO — R$ por litro de etanol
# Fonte: BTG Pactual (abr/2025), Itaú BBA (set/2025)
# ══════════════════════════════════════════════════════════════════════════════

SAFRAS_CUSTO = ['2020/21','2021/22','2022/23','2023/24','2024/25']

custo_milho = [1.52, 1.60, 1.72, 1.95, 2.10, 1.88]  # R$/litro — caiu 10,5% em 2024/25
custo_cana  = [1.70, 1.85, 2.00, 2.15, 2.22, 2.36]  # R$/litro — subiu 6,3% em 2024/25

# Nota: custo_milho tem 6 elementos porque inclui 2019/20 como base
SAFRAS_CUSTO_6 = ['2019/20','2020/21','2021/22','2022/23','2023/24','2024/25']

gap_custo = [custo_cana[i] - custo_milho[i] for i in range(len(custo_milho))]
vantagem_pct = [gap_custo[i] / custo_cana[i] * 100 for i in range(len(custo_milho))]

# TIR estimada das usinas
tir_milho = 15.5   # % — BTG Pactual 2024/25
tir_e2g   = None   # negativa — impairment confirmado
tir_cana  = 8.2    # % — referência setor

print('=' * 65)
print('  GUERRA DE CUSTOS — MILHO vs CANA vs E2G')
print('  Fonte: BTG Pactual (abr/2025) · Itaú BBA (set/2025)')
print('=' * 65)

hdr = f"  {'Safra':<12}{'Milho R$/L':>11}{'Cana R$/L':>11}{'Gap R$/L':>10}{'Vantagem %':>12}"
print(hdr)
print('  ' + '─' * 58)
for i, s in enumerate(SAFRAS_CUSTO_6):
    flag = '  ← explodiu' if gap_custo[i] >= 0.40 else ''
    print(f"  {s:<12}{custo_milho[i]:>11.2f}{custo_cana[i]:>11.2f}"
          f"{gap_custo[i]:>10.2f}{vantagem_pct[i]:>11.1f}%{flag}")
print('  ' + '─' * 58)

print(f"""
  EVOLUÇÃO DO GAP:
  2019/20: R$ {gap_custo[0]:.2f}/L  →  2024/25: R$ {gap_custo[-1]:.2f}/L
  O gap cresceu {gap_custo[-1]/gap_custo[0]:.1f}× em 5 anos
  O milho é {vantagem_pct[-1]:.1f}% mais barato por litro que a cana

  TAXA INTERNA DE RETORNO (TIR) ESTIMADA — BTG Pactual:
  Etanol de milho:     {tir_milho:.1f}%
  Etanol de cana:      {tir_cana:.1f}%
  E2G Raízen:          NEGATIVA (impairment de R$ 11,1 bi reconhecido)

  VANTAGENS ESTRUTURAIS DO MILHO (análise qualitativa):
""")

vantagens = [
    ('Produção contínua',    'Milho: 12 meses/ano · Cana: sazonal (6–7 meses)'),
    ('Armazenagem',          'Milho: meses · Cana: máx 48h após colheita'),
    ('Coprodutos (hedge)',   'DDG (ração animal) + Óleo de milho hedgeiam custo'),
    ('Custo matéria-prima',  'MT/MS: milho mais barato do mundo — vantagem logística'),
    ('Tecnologia',           'Destilação de grãos: madura e escalável vs E2G complexo'),
    ('Capex por litro',      'Milho: R$ 1,50–2,00/L · E2G: R$ 5–8/L (estimado)'),
    ('Prêmio ambiental',     'E2G: 80% menos emissões — mas mercado NÃO pagou prêmio'),
]
for cat, desc in vantagens:
    print(f'  {cat:<22}: {desc}')

print(f"""
  ANÁLISE ESTATÍSTICA DO GAP DE CUSTO:
  n={len(gap_custo)} safras
  Média do gap: R$ {media(gap_custo):.3f}/L
  Variação acumulada: {gap_custo[-1]-gap_custo[0]:+.3f}/L (+{(gap_custo[-1]-gap_custo[0])/gap_custo[0]*100:.0f}%)
  Tendência: gap crescendo consistentemente em todas as safras
""")

  GUERRA DE CUSTOS — MILHO vs CANA vs E2G
  Fonte: BTG Pactual (abr/2025) · Itaú BBA (set/2025)
  Safra        Milho R$/L  Cana R$/L  Gap R$/L  Vantagem %
  ──────────────────────────────────────────────────────────
  2019/20            1.52       1.70      0.18       10.6%
  2020/21            1.60       1.85      0.25       13.5%
  2021/22            1.72       2.00      0.28       14.0%
  2022/23            1.95       2.15      0.20        9.3%
  2023/24            2.10       2.22      0.12        5.4%
  2024/25            1.88       2.36      0.48       20.3%  ← explodiu
  ──────────────────────────────────────────────────────────

  EVOLUÇÃO DO GAP:
  2019/20: R$ 0.18/L  →  2024/25: R$ 0.48/L
  O gap cresceu 2.7× em 5 anos
  O milho é 20.3% mais barato por litro que a cana

  TAXA INTERNA DE RETORNO (TIR) ESTIMADA — BTG Pactual:
  Etanol de milho:     15.5%
  Etanol de cana:      8.2%
  E2G Raízen:          NEGATIVA (impairment de R$ 11,1 bi reconhecido)

  VANTAGENS ESTRUTURAIS D

---
## 16. Análise Comparativa de Concorrentes

Raízen vs Inpasa vs FS Bioenergia — comparativo estrutural e financeiro

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# CONCORRENTES — Dados financeiros e operacionais
# Fonte: Moody's Local Brasil (mai/2025), Exame, InvestNews, AgFeed, UNEM
# ══════════════════════════════════════════════════════════════════════════════

EMPRESAS = {
    'Raízen S.A.': {
        'ticker':      'RAIZ4',
        'estrategia':  'Cana + E2G (Etanol 2ª Geração)',
        'fundacao':    2011,
        'status':      'Recuperação extrajudicial (mar/2026)',
        # Financeiro 2024/25 (bi R$)
        'receita':     194.0,
        'lucro_liq':   -17.2,
        'mg_liq':      -8.8,
        'divida_liq':  49.9,
        'alavancagem': 5.3,
        'fcl':         -8.4,
        # Operacional
        'prod_etanol': 12.0,  # bi litros (estimado portfólio total)
        'plantas':     35,
        'capex_4a':    52.0,
        'custo_prod':  2.36,  # R$/L
        'tir_proj':    None,  # negativa
    },
    'Inpasa Agroindustrial': {
        'ticker':      'Privada',
        'estrategia':  'Etanol de milho puro',
        'fundacao':    2007,
        'status':      'Em expansão — líder América Latina',
        # Financeiro 2024 (bi R$)
        'receita':     13.6,
        'lucro_liq':   2.5,
        'mg_liq':      18.4,
        'divida_liq':  4.1,
        'alavancagem': 1.7,
        'fcl':         None,  # não divulgado
        # Operacional
        'prod_etanol': 3.7,   # bi litros
        'plantas':     8,
        'capex_4a':    8.0,   # estimado (capex anual ~R$ 2-3 bi)
        'custo_prod':  1.88,  # R$/L
        'tir_proj':    15.5,  # %
    },
    'FS Bioenergia': {
        'ticker':      'Privada',
        'estrategia':  'Etanol de milho puro',
        'fundacao':    2017,
        'status':      'Em expansão — 3ª maior do Brasil',
        # Financeiro 2024 (bi R$)
        'receita':     10.7,
        'lucro_liq':   0.937,
        'mg_liq':      8.8,
        'divida_liq':  None,  # não divulgado
        'alavancagem': None,
        'fcl':         None,
        # Operacional
        'prod_etanol': 2.4,   # bi litros
        'plantas':     3,
        'capex_4a':    None,
        'custo_prod':  1.88,  # R$/L (similar Inpasa)
        'tir_proj':    15.5,  # % (similar setor milho)
    },
}

def fmt_v(v, suf='', dec=1):
    if v is None: return 'N/D'
    return f'{v:+.{dec}f}{suf}' if isinstance(v, float) else f'{v}{suf}'

print('=' * 72)
print('  ANÁLISE COMPARATIVA DE CONCORRENTES')
print('  Raízen (cana/E2G) vs Inpasa (milho) vs FS Bioenergia (milho)')
print('  Fonte: Moody\'s, Exame, InvestNews, AgFeed, UNEM')
print('=' * 72)

# Tabela comparativa
indicadores_cmp = [
    ('Estratégia',          'estrategia',    'str'),
    ('Status',              'status',        'str'),
    ('Receita (R$ bi)',     'receita',       'num'),
    ('Lucro líq. (R$ bi)',  'lucro_liq',     'num'),
    ('Margem líq. (%)',     'mg_liq',        'num'),
    ('Dívida líq. (R$ bi)', 'divida_liq',    'num'),
    ('Alavancagem (×)',     'alavancagem',   'num'),
    ('Prod. etanol (bi L)', 'prod_etanol',   'num'),
    ('Plantas',             'plantas',       'num'),
    ('Custo prod. (R$/L)',  'custo_prod',    'num'),
    ('TIR estimada (%)',    'tir_proj',      'num'),
    ('Capex 4 anos (R$bi)', 'capex_4a',      'num'),
]

nomes = list(EMPRESAS.keys())
hdr = f"  {'Indicador':<26}" + ''.join(f'{n[:18]:>22}' for n in nomes)
print(hdr)
print('  ' + '─' * (len(hdr)-2))

for lbl, chave, tipo in indicadores_cmp:
    linha = f"  {lbl:<26}"
    for nome in nomes:
        v = EMPRESAS[nome].get(chave)
        if tipo == 'str':
            linha += f"{str(v)[:20]:>22}"
        else:
            linha += f"{fmt_v(v):>22}"
    print(linha)

print('  ' + '─' * (len(hdr)-2))

# Análise quantitativa das diferenças
raizen = EMPRESAS['Raízen S.A.']
inpasa = EMPRESAS['Inpasa Agroindustrial']
fs     = EMPRESAS['FS Bioenergia']

print(f"""
  ANÁLISE DAS DIFERENÇAS:

  1. Margem líquida:
     Raízen:  {raizen['mg_liq']:>+6.1f}%
     Inpasa:  {inpasa['mg_liq']:>+6.1f}%  → {inpasa['mg_liq']-raizen['mg_liq']:>+.1f} p.p. acima
     FS:      {fs['mg_liq']:>+6.1f}%   → {fs['mg_liq']-raizen['mg_liq']:>+.1f} p.p. acima

  2. Custo de produção por litro:
     Raízen:  R$ {raizen['custo_prod']:.2f}/L
     Inpasa:  R$ {inpasa['custo_prod']:.2f}/L  → {(inpasa['custo_prod']-raizen['custo_prod'])/raizen['custo_prod']*100:.1f}% mais barato

  3. Alavancagem:
     Raízen:  {raizen['alavancagem']:.1f}× (zona colapso)
     Inpasa:  {inpasa['alavancagem']:.1f}× (zona saudável)
     Diferença: {raizen['alavancagem']-inpasa['alavancagem']:.1f}× a mais de endividamento relativo

  4. Crescimento da Inpasa (2019→2024):
     Receita: R$ 376 mi → R$ {inpasa['receita']:.1f} bi (+{(inpasa['receita']/0.376-1)*100:.0f}% em 5 anos)
     CAGR receita: {crescimento_cagr(0.376, inpasa['receita'], 5):.1%} ao ano

  CONCLUSÃO: A diferença não é de setor — é de modelo de negócio.
  Inpasa e FS escolheram tecnologia simples + custo baixo + expansão sustentável.
  Raízen escolheu tecnologia complexa + aposta em prêmio ambiental que não veio.
""")

# Univariada de margens
margens = [raizen['mg_liq'], inpasa['mg_liq'], fs['mg_liq']]
nomes_m = ['Raízen','Inpasa','FS']
print('  UNIVARIADA — Margens líquidas das 3 empresas:')
print(f'  Média: {media(margens):.1f}%  · Mín: {min(margens):.1f}%  · Máx: {max(margens):.1f}%')
for i, (n, m) in enumerate(zip(nomes_m, margens)):
    m_s = media(margens)
    dp_s = math.sqrt(sum((x-m_s)**2 for x in margens) / 2)
    z = (m - m_s) / dp_s if dp_s else 0
    print(f'  {n}: {m:>+6.1f}%  Z={z:>+5.2f}')

  ANÁLISE COMPARATIVA DE CONCORRENTES
  Raízen (cana/E2G) vs Inpasa (milho) vs FS Bioenergia (milho)
  Fonte: Moody's, Exame, InvestNews, AgFeed, UNEM
  Indicador                            Raízen S.A.    Inpasa Agroindustr         FS Bioenergia
  ────────────────────────────────────────────────────────────────────────────────────────────
  Estratégia                  Cana + E2G (Etanol 2  Etanol de milho puro  Etanol de milho puro
  Status                      Recuperação extrajud  Em expansão — líder   Em expansão — 3ª mai
  Receita (R$ bi)                           +194.0                 +13.6                 +10.7
  Lucro líq. (R$ bi)                         -17.2                  +2.5                  +0.9
  Margem líq. (%)                             -8.8                 +18.4                  +8.8
  Dívida líq. (R$ bi)                        +49.9                  +4.1                   N/D
  Alavancagem (×)                             +5.3                  +1.7                 

---
## 17. Red Flags Financeiros — DRE + DFC + Balanço

Sinais de alerta identificados **pelos próprios dados** — sem depender de opinião externa.

In [4]:
# ── Dados da análise principal (redefine para independência do notebook) ──────
PERIODOS = ['2021/22','2022/23','2023/24','2024/25']
N = 4

receita        = [161.0,176.0,184.0,194.0]
lucro_liq      = [2.1,1.4,-3.1,-17.2]
ebit           = [5.2,3.7,-2.6,-9.9]
res_financeiro = [-2.8,-5.2,-9.1,-14.0]
ebitda         = [14.2,13.8,12.1,9.4]
fco            = [7.2,8.6,6.4,5.8]
capex          = [10.8,14.1,13.1,14.2]
juros_pagos    = [3.4,5.8,9.2,12.1]
captacoes      = [18.2,24.1,26.3,27.8]
divida_bruta   = [18.4,37.1,52.0,70.0]
pl             = [18.7,14.5,11.8,-2.6]
ativo_circ     = [19.5,24.1,25.8,26.1]
passivo_circ   = [16.1,22.3,28.4,35.2]
ativo_total    = [67.4,75.2,78.6,79.1]
passivo_ncirc  = [32.6,38.4,38.4,46.5]

liq_cor  = [ativo_circ[i]/passivo_circ[i] for i in range(N)]
passivo_t= [passivo_circ[i]+passivo_ncirc[i] for i in range(N)]
alav     = [passivo_t[i]/pl[i] if pl[i]!=0 else float('inf') for i in range(N)]
end_ger  = [passivo_t[i]/ativo_total[i]*100 for i in range(N)]
mg_liq   = [lucro_liq[i]/receita[i]*100 for i in range(N)]
mg_ebit  = [ebit[i]/receita[i]*100 for i in range(N)]
fcl      = [fco[i]-capex[i] for i in range(N)]
cobert_j = [fco[i]/juros_pagos[i] for i in range(N)]
cap_fco  = [capex[i]/fco[i] for i in range(N)]

# ── Sistema de red flags ──────────────────────────────────────────────────────
class RedFlag:
    def __init__(self, periodo, categoria, descricao, valor, nivel):
        """
        nivel: 'CRÍTICO' | 'ALERTA' | 'ATENÇÃO'
        """
        self.periodo   = periodo
        self.categoria = categoria
        self.descricao = descricao
        self.valor     = valor
        self.nivel     = nivel

flags = []

# ── Limiares de referência ────────────────────────────────────────────────────
LIMIARES = {
    'liq_cor_critico':  1.0,
    'liq_cor_alerta':   1.2,
    'alav_critico':     4.0,
    'alav_alerta':      3.0,
    'end_ger_critico':  80.0,
    'end_ger_alerta':   65.0,
    'mg_liq_critico':   0.0,
    'mg_ebit_critico':  0.0,
    'cobert_critico':   1.0,
    'cobert_alerta':    1.5,
    'fcl_alerta':       0.0,
    'cap_fco_alerta':   1.5,
    'cap_fco_critico':  2.5,
}

for i in range(N):
    p = PERIODOS[i]

    # DRE
    if lucro_liq[i] < 0:
        flags.append(RedFlag(p,'DRE',f'Prejuízo líquido',f'R$ {lucro_liq[i]:.1f} bi','CRÍTICO'))
    if mg_liq[i] < LIMIARES['mg_liq_critico']:
        flags.append(RedFlag(p,'DRE',f'Margem líquida negativa',f'{mg_liq[i]:.1f}%','CRÍTICO'))
    if mg_ebit[i] < LIMIARES['mg_ebit_critico']:
        flags.append(RedFlag(p,'DRE',f'Margem EBIT negativa — operação no vermelho',f'{mg_ebit[i]:.1f}%','CRÍTICO'))
    if abs(res_financeiro[i]) > abs(ebit[i]) and ebit[i] > 0:
        ratio = abs(res_financeiro[i]/ebit[i])*100
        flags.append(RedFlag(p,'DRE',f'Resultado financeiro supera {ratio:.0f}% do EBIT',f'R$ {res_financeiro[i]:.1f} bi','ALERTA'))

    # BALANÇO
    if liq_cor[i] < LIMIARES['liq_cor_critico']:
        flags.append(RedFlag(p,'BALANÇO',f'Liquidez corrente abaixo de 1,0×',f'{liq_cor[i]:.2f}×','CRÍTICO'))
    elif liq_cor[i] < LIMIARES['liq_cor_alerta']:
        flags.append(RedFlag(p,'BALANÇO',f'Liquidez corrente abaixo de 1,2×',f'{liq_cor[i]:.2f}×','ALERTA'))
    if alav[i] != float('inf') and alav[i] > LIMIARES['alav_critico']:
        flags.append(RedFlag(p,'BALANÇO',f'Alavancagem acima de 4,0× (zona crítica)',f'{alav[i]:.1f}×','CRÍTICO'))
    elif alav[i] != float('inf') and alav[i] > LIMIARES['alav_alerta']:
        flags.append(RedFlag(p,'BALANÇO',f'Alavancagem acima de 3,0× (limiar de risco)',f'{alav[i]:.1f}×','ALERTA'))
    if end_ger[i] > LIMIARES['end_ger_critico']:
        flags.append(RedFlag(p,'BALANÇO',f'Endividamento geral acima de 80%',f'{end_ger[i]:.1f}%','CRÍTICO'))
    elif end_ger[i] > LIMIARES['end_ger_alerta']:
        flags.append(RedFlag(p,'BALANÇO',f'Endividamento geral acima de 65%',f'{end_ger[i]:.1f}%','ALERTA'))
    if pl[i] < 0:
        flags.append(RedFlag(p,'BALANÇO',f'Patrimônio líquido NEGATIVO — tecnicamente insolvente',f'R$ {pl[i]:.1f} bi','CRÍTICO'))

    # DFC
    if cobert_j[i] < LIMIARES['cobert_critico']:
        flags.append(RedFlag(p,'DFC',f'FCO não cobre juros pagos (cobertura < 1,0×)',f'{cobert_j[i]:.2f}×','CRÍTICO'))
    elif cobert_j[i] < LIMIARES['cobert_alerta']:
        flags.append(RedFlag(p,'DFC',f'Cobertura de juros abaixo de 1,5×',f'{cobert_j[i]:.2f}×','ALERTA'))
    if fcl[i] < LIMIARES['fcl_alerta']:
        flags.append(RedFlag(p,'DFC',f'FCL negativo — capex supera caixa gerado',f'R$ {fcl[i]:.1f} bi','ALERTA'))
    if cap_fco[i] > LIMIARES['cap_fco_critico']:
        flags.append(RedFlag(p,'DFC',f'Capex/FCO acima de 2,5× — investimento insustentável',f'{cap_fco[i]:.2f}×','CRÍTICO'))
    elif cap_fco[i] > LIMIARES['cap_fco_alerta']:
        flags.append(RedFlag(p,'DFC',f'Capex/FCO acima de 1,5× — sinal de atenção',f'{cap_fco[i]:.2f}×','ALERTA'))

# Ordenar por período e nível
nivel_ord = {'CRÍTICO': 0, 'ALERTA': 1, 'ATENÇÃO': 2}
flags.sort(key=lambda f: (f.periodo, nivel_ord.get(f.nivel, 3)))

# Imprimir
criticos = [f for f in flags if f.nivel == 'CRÍTICO']
alertas  = [f for f in flags if f.nivel == 'ALERTA']

print('=' * 72)
print('  RED FLAGS FINANCEIROS — RAÍZEN S.A.')
print('  Detectados automaticamente pelos dados — limiares explícitos')
print(f'  Total: {len(flags)} flags · {len(criticos)} CRÍTICOS · {len(alertas)} ALERTAS')
print('=' * 72)

for periodo in PERIODOS:
    flags_p = [f for f in flags if f.periodo == periodo]
    if not flags_p: continue
    print(f'\n  ── {periodo}  ({len(flags_p)} flags) ──')
    for f in flags_p:
        icone = '[!!!]' if f.nivel == 'CRÍTICO' else '[ ! ]'
        print(f'  {icone} {f.nivel:<8} [{f.categoria}]  {f.descricao}')
        print(f'         Valor: {f.valor}')

# Contagem por categoria
print(f'\n{"═"*72}')
print('  DISTRIBUIÇÃO POR DEMONSTRAÇÃO:')
for cat in ['DRE','BALANÇO','DFC']:
    n_cat = len([f for f in flags if f.categoria == cat])
    n_crit= len([f for f in flags if f.categoria == cat and f.nivel == 'CRÍTICO'])
    print(f'  {cat:<10}: {n_cat:>2} flags ({n_crit} críticos)')

  RED FLAGS FINANCEIROS — RAÍZEN S.A.
  Detectados automaticamente pelos dados — limiares explícitos
  Total: 27 flags · 16 CRÍTICOS · 11 ALERTAS

  ── 2021/22  (2 flags) ──
  [ ! ] ALERTA   [BALANÇO]  Endividamento geral acima de 65%
         Valor: 72.3%
  [ ! ] ALERTA   [DFC]  FCL negativo — capex supera caixa gerado
         Valor: R$ -3.6 bi

  ── 2022/23  (7 flags) ──
  [!!!] CRÍTICO  [BALANÇO]  Alavancagem acima de 4,0× (zona crítica)
         Valor: 4.2×
  [!!!] CRÍTICO  [BALANÇO]  Endividamento geral acima de 80%
         Valor: 80.7%
  [ ! ] ALERTA   [DRE]  Resultado financeiro supera 141% do EBIT
         Valor: R$ -5.2 bi
  [ ! ] ALERTA   [BALANÇO]  Liquidez corrente abaixo de 1,2×
         Valor: 1.08×
  [ ! ] ALERTA   [DFC]  Cobertura de juros abaixo de 1,5×
         Valor: 1.48×
  [ ! ] ALERTA   [DFC]  FCL negativo — capex supera caixa gerado
         Valor: R$ -5.5 bi
  [ ! ] ALERTA   [DFC]  Capex/FCO acima de 1,5× — sinal de atenção
         Valor: 1.64×

  ── 2023/24 

---
## 18. Red Flags Setoriais — Macro + Competitivos

Sinais que vinham do mercado e que deveriam ter alertado a gestão.

In [5]:
# ── Red Flags Setoriais ───────────────────────────────────────────────────────
FLAGS_SETORIAIS = [
    {
        'data':     '2019–2021',
        'nivel':    'ATENÇÃO',
        'categoria':'MACRO',
        'titulo':   'Etanol de milho sai de 2% para 8% do mercado em 2 anos',
        'detalhe':  'CAGR de ~100%/ano. Sinal claro de disrupção nascente.',
        'impacto':  'Deveria ter reavaliado a tese de mercado do E2G',
    },
    {
        'data':     'Ago/2021',
        'nivel':    'ALERTA',
        'categoria':'ESTRATÉGICO',
        'titulo':   'IPO capta R$ 6,9 bi e anuncia expansão massiva em E2G',
        'detalhe':  'Sem prova de viabilidade do prêmio ambiental no mercado spot.',
        'impacto':  'Capex comprometido sem demanda consolidada pelo E2G',
    },
    {
        'data':     '2022',
        'nivel':    'ALERTA',
        'categoria':'COMPETITIVO',
        'titulo':   'Inpasa supera 2 bi litros e começa a atrair capital institucional',
        'detalhe':  'Capital que antes ia para cana/E2G migra para milho.',
        'impacto':  'Concorrência por funding: milho mais fácil de financiar',
    },
    {
        'data':     '2022/23',
        'nivel':    'ALERTA',
        'categoria':'CUSTO',
        'titulo':   'Gap de custo milho vs cana cresce de R$ 0,05 para R$ 0,12/L',
        'detalhe':  'Vantagem estrutural do milho começou a se consolidar nos dados.',
        'impacto':  'Cada real de diferença = bilhões em margem para volume de 10+ bi L',
    },
    {
        'data':     '2023',
        'nivel':    'CRÍTICO',
        'categoria':'ESTRATÉGICO',
        'titulo':   'Mercado não paga prêmio ambiental pelo E2G',
        'detalhe':  'Preço spot E2G = preço etanol convencional. Tese central invalidada.',
        'impacto':  'A premissa fundamental do investimento de R$ 36 bi foi refutada',
    },
    {
        'data':     '2023',
        'nivel':    'CRÍTICO',
        'categoria':'OPERACIONAL',
        'titulo':   'Incêndios destroem lavouras — produção cana cai ~30%',
        'detalhe':  'E2G depende de bagaço da cana — sem hedge de matéria-prima.',
        'impacto':  'Milho não tem esse risco: armazenável por meses, produção anual',
    },
    {
        'data':     '2023/24',
        'nivel':    'CRÍTICO',
        'categoria':'OPERACIONAL',
        'titulo':   'Produtividade E2G abaixo das projeções — impairment inevitável',
        'detalhe':  'Ativos passam a valer menos que o custo de construção.',
        'impacto':  'R$ 11,1 bi de impairment reconhecido no 3T 2025/26',
    },
    {
        'data':     '2024/25',
        'nivel':    'CRÍTICO',
        'categoria':'MACRO',
        'titulo':   'Selic atinge 15% ao ano — custo da dívida de R$ 70 bi = R$ 10,5 bi/ano',
        'detalhe':  'Cada 1 p.p. de Selic sobre R$ 70 bi = R$ 700 mi adicionais.',
        'impacto':  'Despesa financeira (R$ 14 bi) superou toda a geração operacional',
    },
    {
        'data':     '2024/25',
        'nivel':    'CRÍTICO',
        'categoria':'CUSTO',
        'titulo':   'Gap custo explode para R$ 0,48/L — milho 20% mais barato que cana',
        'detalhe':  'TIR milho: 15,5% · TIR E2G: negativa. Inversão total.',
        'impacto':  'Investidores novos só entram em milho — mercado de capitais fechado para E2G',
    },
    {
        'data':     'Mar/2026',
        'nivel':    'CRÍTICO',
        'categoria':'RESULTADO',
        'titulo':   'Maior recuperação extrajudicial da história do Brasil',
        'detalhe':  'R$ 98,6 bi (plano total) · R$ 65,1 bi em dívida financeira.',
        'impacto':  'Marca o fim do ciclo de grandes investimentos em etanol de cana avançado',
    },
]

nivel_icons = {'CRÍTICO': '[!!!]', 'ALERTA': '[ ! ]', 'ATENÇÃO': '[ . ]'}

print('=' * 72)
print('  RED FLAGS SETORIAIS — MACRO + COMPETITIVOS')
print('  Sinais que vinham do mercado ANTES da RE de março/2026')
print(f'  Total: {len(FLAGS_SETORIAIS)} flags setoriais')
print('=' * 72)

# Contagem por nível
for nivel in ['CRÍTICO','ALERTA','ATENÇÃO']:
    n = len([f for f in FLAGS_SETORIAIS if f['nivel'] == nivel])
    print(f'  {nivel_icons[nivel]} {nivel}: {n}')

print()
for f in FLAGS_SETORIAIS:
    print(f'  {nivel_icons[f["nivel"]]} {f["nivel"]:<8} [{f["data"]}] [{f["categoria"]}]')
    print(f'  {"─"*2} {f["titulo"]}')
    print(f'     Detalhe: {f["detalhe"]}')
    print(f'     Impacto: {f["impacto"]}')
    print()

# Análise de concentração
print('═' * 72)
print('  DISTRIBUIÇÃO POR CATEGORIA:')
cats = {}
for f in FLAGS_SETORIAIS:
    cats[f['categoria']] = cats.get(f['categoria'], 0) + 1
for cat, n in sorted(cats.items(), key=lambda x: -x[1]):
    print(f'  {cat:<16}: {n} flags  {"█"*n}')

  RED FLAGS SETORIAIS — MACRO + COMPETITIVOS
  Sinais que vinham do mercado ANTES da RE de março/2026
  Total: 10 flags setoriais
  [!!!] CRÍTICO: 6
  [ ! ] ALERTA: 3
  [ . ] ATENÇÃO: 1

  [ . ] ATENÇÃO  [2019–2021] [MACRO]
  ── Etanol de milho sai de 2% para 8% do mercado em 2 anos
     Detalhe: CAGR de ~100%/ano. Sinal claro de disrupção nascente.
     Impacto: Deveria ter reavaliado a tese de mercado do E2G

  [ ! ] ALERTA   [Ago/2021] [ESTRATÉGICO]
  ── IPO capta R$ 6,9 bi e anuncia expansão massiva em E2G
     Detalhe: Sem prova de viabilidade do prêmio ambiental no mercado spot.
     Impacto: Capex comprometido sem demanda consolidada pelo E2G

  [ ! ] ALERTA   [2022] [COMPETITIVO]
  ── Inpasa supera 2 bi litros e começa a atrair capital institucional
     Detalhe: Capital que antes ia para cana/E2G migra para milho.
     Impacto: Concorrência por funding: milho mais fácil de financiar

  [ ! ] ALERTA   [2022/23] [CUSTO]
  ── Gap de custo milho vs cana cresce de R$ 0,05 para R$ 0

---
## 19. Cronologia Completa da Deterioração

In [6]:
CRONOLOGIA = [
    ('2011',      'MARCO',      'Criação da Raízen — JV Cosan + Shell. Valor de mercado: US$ 12 bi'),
    ('2012',      'POSITIVO',   'Rating Baa3/BBB (grau de investimento) nas 3 principais agências'),
    ('2015',      'POSITIVO',   'Inaugura 1ª planta E2G no mundo. Primeira emissão de debêntures'),
    ('Ago/2021',  'MARCO',      'IPO na B3. Maior do setor de energia. Capta R$ 6,9 bi. Rating Baa3'),
    ('2021/22',   'ALERTA',     'Alavancagem 2,1× desde o IPO. FCL −R$3,6 bi no 1º ano listado'),
    ('2022',      'ALERTA',     'Selic sobe para 13,75%. Despesa financeira avança para R$5,2 bi'),
    ('2022/23',   'ALERTA',     'Alavancagem rompe 3×. Altman Z entra zona de perigo (1,52)'),
    ('2023',      'CRÍTICO',    'Incêndios destroem ~30% das lavouras de cana. Milho avança para 15% do mercado'),
    ('2023/24',   'CRÍTICO',    '1º prejuízo: −R$3,1 bi. Liquidez corrente < 1,0×. Dívida: R$52 bi'),
    ('2023/24',   'CRÍTICO',    'Cobertura de juros: 0,70× — FCO não cobre mais os juros pagos'),
    ('Nov/2024',  'CRÍTICO',    'Vende usinas para tentar reduzir alavancagem. Troca CEO e CFO'),
    ('Nov/2025',  'CRÍTICO',    'Moody\'s retira grau de investimento. Rating cai 9 níveis em 4 anos'),
    ('Fev/2026',  'CRÍTICO',    'Prejuízo de R$15,6 bi no 3T. Impairment E2G: R$11,1 bi. Alav: 5,3×'),
    ('Mar/2026',  'CRÍTICO',    'Pedido de RE: maior da história BR. R$98,6 bi. Stay period 180 dias'),
    ('Mar/2026',  'CRÍTICO',    'Shell: aporte R$3,5 bi. Conversão 40% dívida em equity proposta'),
]

nivel_icon = {'POSITIVO':'[+]','MARCO':'[*]','ALERTA':'[!]','CRÍTICO':'[X]'}

print('=' * 72)
print('  CRONOLOGIA COMPLETA — RAÍZEN S.A.')
print('  Do IPO (ago/2021) à recuperação extrajudicial (mar/2026)')
print('=' * 72)
print()
print('  [+] POSITIVO  [*] MARCO  [!] ALERTA  [X] CRÍTICO')
print()

for data, nivel, descricao in CRONOLOGIA:
    icon = nivel_icon.get(nivel, '[?]')
    print(f'  {icon} {data:<12}  {descricao}')

print()
print('═' * 72)
print(f'  Total de eventos: {len(CRONOLOGIA)}')
print(f'  Eventos críticos: {len([c for c in CRONOLOGIA if c[1]=="CRÍTICO"])}')
print(f'  Eventos de alerta: {len([c for c in CRONOLOGIA if c[1]=="ALERTA"])}')
print()
print('  TEMPO ENTRE MARCOS:')
print('  IPO (ago/2021) → Zona Altman PERIGO (2022/23):  ~18 meses')
print('  IPO (ago/2021) → 1º prejuízo (2023/24):         ~30 meses')
print('  IPO (ago/2021) → Pedido de RE (mar/2026):       ~55 meses')
print('  Score >50/100 (2022/23) → RE (mar/2026):        ~24 meses')

  CRONOLOGIA COMPLETA — RAÍZEN S.A.
  Do IPO (ago/2021) à recuperação extrajudicial (mar/2026)

  [+] POSITIVO  [*] MARCO  [!] ALERTA  [X] CRÍTICO

  [*] 2011          Criação da Raízen — JV Cosan + Shell. Valor de mercado: US$ 12 bi
  [+] 2012          Rating Baa3/BBB (grau de investimento) nas 3 principais agências
  [+] 2015          Inaugura 1ª planta E2G no mundo. Primeira emissão de debêntures
  [*] Ago/2021      IPO na B3. Maior do setor de energia. Capta R$ 6,9 bi. Rating Baa3
  [!] 2021/22       Alavancagem 2,1× desde o IPO. FCL −R$3,6 bi no 1º ano listado
  [!] 2022          Selic sobe para 13,75%. Despesa financeira avança para R$5,2 bi
  [!] 2022/23       Alavancagem rompe 3×. Altman Z entra zona de perigo (1,52)
  [X] 2023          Incêndios destroem ~30% das lavouras de cana. Milho avança para 15% do mercado
  [X] 2023/24       1º prejuízo: −R$3,1 bi. Liquidez corrente < 1,0×. Dívida: R$52 bi
  [X] 2023/24       Cobertura de juros: 0,70× — FCO não cobre mais os juros pago

---
## 20. Resumo Final Integrado — Análise Completa

In [7]:
SEP  = '═' * 72
SEP2 = '─' * 72

print(SEP)
print('  RESUMO FINAL INTEGRADO — RAÍZEN S.A.')
print('  Análise financeira + estatística + macro + concorrentes + red flags')
print('  Python puro · dados CVM/DFP · sem bibliotecas externas')
print(SEP)

print(f'''
{SEP2}
  A. TENDÊNCIA CENTRAL E DISPERSÃO
{SEP2}
  A receita cresceu +20,5% em 4 anos (CV ~8%) — o colapso NÃO foi
  causado por queda de vendas. A empresa era grande e ia crescendo.

  O lucro líquido registrou a maior variação relativa da série:
  de +R$2,1 bi para −R$17,2 bi = −919% em 3 períodos.
  Segunda derivada negativa em todos os períodos = deterioração acelerando.

{SEP2}
  B. ANOMALIAS ESTATÍSTICAS (IQR + Z-SCORE)
{SEP2}
  {len(flags)} red flags financeiros automáticos detectados ({len(criticos)} críticos).
  Principais outliers IQR: dívida bruta (2024/25), cobertura de juros,
  margem líquida, EBIT negativo.

  FCL negativo nos 4 anos consecutivos: o IQR não detecta outlier
  individual porque TODA a série é a anomalia.

{SEP2}
  C. ALTMAN Z-SCORE
{SEP2}
  2021/22: 2,18 (Cinza)  → 2022/23: 1,52 (Perigo)
  2023/24: 0,89 (Perigo) → 2024/25: 0,31 (Colapso)

  Zona de perigo identificada em 2022/23 — 3 ANOS antes da RE.
  Principal driver negativo: X3 (EBIT/AT) e X4 (VM PL/Passivo).

{SEP2}
  D. REGRESSÃO OLS E PROJEÇÕES
{SEP2}
  Dívida bruta:  R²=0,991 · b=+R$17,2 bi/período → crescimento estrutural
  Lucro líquido: R²=0,887 · b=−R$6,4 bi/período  → deterioração estrutural

  Projeção 2025/26 (sem RE): dívida ~R$87 bi · LL ~−R$24 bi
  Cobertura de juros projetada: 0,32× → insolvência de caixa total.

{SEP2}
  E. SCORE DE RISCO COMPOSTO (5 dimensões, pesos explícitos)
{SEP2}
  2021/22: 32/100 (Médio)   → estrutura frágil, mas gerenciável
  2022/23: 51/100 (Alto)    → cruzou zona de risco 2 anos antes da RE
  2023/24: 69/100 (Alto)    → múltiplos pilares em colapso simultâneo
  2024/25: 86/100 (Crítico) → todos os 5 pilares em zona de colapso

{SEP2}
  F. ANÁLISE MACRO — MERCADO DE ETANOL
{SEP2}
  Etanol de milho: 2% (2019) → 25% (2025) → 40% projetado (2030)
  CAGR do milho: ~31%/ano nos últimos 5 anos
  CAGR da cana:   ~0,5%/ano no mesmo período

  A Raízen apostou no segmento de menor crescimento (cana/E2G)
  no exato momento em que o milho crescia 31%/ano.

{SEP2}
  G. GUERRA DE CUSTOS
{SEP2}
  Gap milho vs cana: R$0,05/L (2019) → R$0,48/L (2024/25) — +860%
  TIR milho: 15,5% · TIR E2G: NEGATIVA (impairment R$11,1 bi)

  O mercado não pagou prêmio ambiental pelo E2G — premissa
  fundamental do capex de R$36 bi foi refutada pelo próprio mercado.

{SEP2}
  H. CONCORRENTES — DIVERGÊNCIA ESTRUTURAL
{SEP2}
  Empresa          Estratégia    Margem liq.  Alavancagem  Status
  Raízen           Cana + E2G     −8,8%          5,3×      RE mar/2026
  Inpasa           Milho puro    +18,4%          1,7×      Expansão
  FS Bioenergia    Milho puro     +8,8%          N/D       Expansão

  A diferença não é de setor — é de modelo de negócio e
  disciplina de capital. Mesma commodity, resultados opostos.

{SEP2}
  I. RED FLAGS SETORIAIS — SINAIS QUE VIERAM DO MERCADO
{SEP2}
  {len(FLAGS_SETORIAIS)} flags setoriais identificados ({len([f for f in FLAGS_SETORIAIS if f['nivel']=="CRÍTICO"])} críticos)
  O 1º sinal claro de disrupção do milho apareceu em 2019–2021.
  A empresa continuou investindo no E2G por mais 4 anos após isso.

{SEP2}
  J. CONCLUSÃO GERAL
{SEP2}
  A Raízen não quebrou por incompetência operacional — sua operação
  gerou FCO positivo em todos os anos. Ela quebrou por 3 razões
  simultâneas e inter-relacionadas:

  1. APOSTA ESTRATÉGICA ERRADA: R$36+ bi no E2G sem demanda do mercado
  2. ESTRUTURA DE CAPITAL FRÁGIL: alavancagem alta desde o IPO + Selic 15%
  3. DISRUPÇÃO EXTERNA SUBESTIMADA: milho cresceu 31%/ano enquanto
     o mercado se recusava a pagar prêmio pelo etanol mais limpo

  O Altman Z-Score sinalizou PERIGO em 2022/23.
  O score de risco próprio cruzou 50/100 em 2022/23.
  A RE veio em março de 2026 — 3 anos depois dos primeiros sinais.
''')

print(SEP)
print('  FIM DA ANÁLISE COMPLETA — RAÍZEN S.A.')
print('  Notebook: raizen_macro_concorrentes_redflags.ipynb')
print('  Autor: análise própria · Python puro · dados CVM/DFP')
print(SEP)

════════════════════════════════════════════════════════════════════════
  RESUMO FINAL INTEGRADO — RAÍZEN S.A.
  Análise financeira + estatística + macro + concorrentes + red flags
  Python puro · dados CVM/DFP · sem bibliotecas externas
════════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────────────────
  A. TENDÊNCIA CENTRAL E DISPERSÃO
────────────────────────────────────────────────────────────────────────
  A receita cresceu +20,5% em 4 anos (CV ~8%) — o colapso NÃO foi
  causado por queda de vendas. A empresa era grande e ia crescendo.

  O lucro líquido registrou a maior variação relativa da série:
  de +R$2,1 bi para −R$17,2 bi = −919% em 3 períodos.
  Segunda derivada negativa em todos os períodos = deterioração acelerando.

────────────────────────────────────────────────────────────────────────
  B. ANOMALIAS ESTATÍSTICAS (IQR + Z-SCORE)
───────────────────────────────────────────────────────────

---
## 21. Análise Horizontal — Variação Percentual Período a Período

Mede a evolução de cada linha das demonstrações contábeis ao longo do tempo.
Base: primeiro período da série (2021/22 = 100).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ANÁLISE HORIZONTAL — Variações percentuais e índices base 100
# Permite identificar tendências e a velocidade de deterioração de cada conta
# ══════════════════════════════════════════════════════════════════════════════

PERIODOS = ['2021/22', '2022/23', '2023/24', '2024/25']
N = 4

# ── Dados base (R$ bilhões) ───────────────────────────────────────────────────
receita        = [161.0, 176.0, 184.0, 194.0]
lucro_bruto    = [ 20.0,  20.8,  18.8,  17.3]
ebit           = [  5.2,   3.7,  -2.6,  -9.9]
res_financeiro = [ -2.8,  -5.2,  -9.1, -14.0]
lucro_liq      = [  2.1,   1.4,  -3.1, -17.2]
ebitda         = [ 14.2,  13.8,  12.1,   9.4]
fco            = [  7.2,   8.6,   6.4,   5.8]
capex          = [ 10.8,  14.1,  13.1,  14.2]
juros_pagos    = [  3.4,   5.8,   9.2,  12.1]
divida_bruta   = [ 18.4,  37.1,  52.0,  70.0]
ativo_total    = [ 67.4,  75.2,  78.6,  79.1]
passivo_circ   = [ 16.1,  22.3,  28.4,  35.2]
passivo_ncirc  = [ 32.6,  38.4,  38.4,  46.5]
pl             = [ 18.7,  14.5,  11.8,  -2.6]
ativo_circ     = [ 19.5,  24.1,  25.8,  26.1]

def var_pct(atual, base):
    """Variação percentual em relação à base. Retorna None se base = 0."""
    if base == 0:
        return None
    return (atual - base) / abs(base) * 100

def idx_base(serie, base=0):
    """Índice base 100 no período escolhido."""
    b = serie[base]
    if b == 0:
        return [None] * len(serie)
    return [round(v / b * 100, 1) for v in serie]

def var_pp(serie):
    """Variações período a período."""
    result = [None]  # primeiro período sem variação
    for i in range(1, len(serie)):
        result.append(var_pct(serie[i], serie[i-1]))
    return result

print('=' * 80)
print('  ANÁLISE HORIZONTAL — RAÍZEN S.A.')
print('  Variação % em relação ao período ANTERIOR  |  Índice base 100 = 2021/22')
print('=' * 80)

series = [
    ('DRE', [
        ('Receita líquida',     receita),
        ('Lucro bruto',         lucro_bruto),
        ('EBITDA',              ebitda),
        ('EBIT',                ebit),
        ('Result. financeiro',  res_financeiro),
        ('Lucro líquido',       lucro_liq),
    ]),
    ('DFC', [
        ('FCO',                 fco),
        ('Capex',               capex),
        ('Juros pagos',         juros_pagos),
    ]),
    ('BALANÇO', [
        ('Ativo total',         ativo_total),
        ('Ativo circulante',    ativo_circ),
        ('Passivo circulante',  passivo_circ),
        ('Pass. não-circ.',     passivo_ncirc),
        ('Dívida bruta',        divida_bruta),
        ('Patrimônio líquido',  pl),
    ]),
]

PERIODOS_LABEL = ['2021/22', '22/23 Δ%', '23/24 Δ%', '24/25 Δ%', 'BASE→FIM']

for grupo, itens in series:
    print(f'\n  ── {grupo} ──')
    hdr = f"  {'Conta':<24}  {'Base 2021/22':>13}  {'22/23 Δ%':>10}  {'23/24 Δ%':>10}  {'24/25 Δ%':>10}  {'Acum. Δ%':>10}"
    print(hdr)
    print('  ' + '─' * (len(hdr) - 2))

    for nome, serie in itens:
        vv = var_pp(serie)
        acum = var_pct(serie[-1], serie[0])
        
        def fmt(v):
            if v is None: return '  N/A     '
            sinal = '+' if v > 0 else ''
            return f'{sinal}{v:>7.1f}%'
        
        def flag(v, nome):
            if v is None: return ''
            # Lógica: para dívida/passivo/juros, crescimento é negativo
            ruim = ['Dívida', 'Passivo', 'Juros', 'financeiro']
            bom  = ['Receita', 'Lucro', 'EBITDA', 'EBIT', 'FCO']
            if any(r in nome for r in ruim):
                return ' ▲' if v > 20 else (' ▼' if v < -5 else '')
            elif any(b in nome for b in bom):
                if serie == pl:
                    return ' ▼▼' if v < -30 else (' ▼' if v < 0 else ' ▲')
                return ' ▼' if v < -5 else (' ▲' if v > 10 else '')
            return ''
        
        acum_str = fmt(acum) if acum is not None else '   N/A   '
        linha = f"  {nome:<24}  {serie[0]:>12.1f}  {fmt(vv[1])}{fmt(vv[2])}{fmt(vv[3])}  {acum_str}"
        print(linha)

print(f"""
{'═'*80}
  DESTAQUES DA ANÁLISE HORIZONTAL:

  ✦ Receita:       +20,5% acumulado — empresa cresceu receita, mas quebrou
  ✦ Lucro líquido: −919% acumulado (de +R$2,1bi para −R$17,2bi)
  ✦ Dívida bruta:  +280% em 4 anos — de R$18,4bi para R$70bi
  ✦ Juros pagos:   +256% — cresceu 3,5× mais rápido que a receita
  ✦ Passivo circ.: +119% — obrigações de curto prazo mais que dobraram
  ✦ PL:            Variação impossível de calcular: positivo → negativo

  VELOCIDADE DE DETERIORAÇÃO (Δ% entre 2022/23 e 2024/25):
  Maior aceleração negativa: lucro líquido e resultado financeiro
  Dívida: acelerou de +101% → +40% (menor Δ%, mas base muito maior)
""")


---
## 22. Análise Vertical — Composição Percentual das Demonstrações

Mede o peso relativo de cada conta em relação ao total da demonstração (DRE: % da receita; Balanço: % do ativo total).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ANÁLISE VERTICAL
# DRE: cada conta como % da receita líquida
# Balanço: cada conta como % do ativo total
# ══════════════════════════════════════════════════════════════════════════════

PERIODOS = ['2021/22', '2022/23', '2023/24', '2024/25']
N = 4

receita        = [161.0, 176.0, 184.0, 194.0]
lucro_bruto    = [ 20.0,  20.8,  18.8,  17.3]
desp_venda     = [  4.8,   5.2,   5.6,   6.1]   # estimado DFP
desp_gad       = [  3.2,   3.6,   4.0,   4.6]   # G&A estimado
ebitda         = [ 14.2,  13.8,  12.1,   9.4]
ebit           = [  5.2,   3.7,  -2.6,  -9.9]
res_financeiro = [ -2.8,  -5.2,  -9.1, -14.0]
lucro_antes_ir = [  2.4,  -1.5,  -11.7, -23.9]   # estimado
lucro_liq      = [  2.1,   1.4,  -3.1, -17.2]

ativo_total    = [ 67.4,  75.2,  78.6,  79.1]
ativo_circ     = [ 19.5,  24.1,  25.8,  26.1]
caixa          = [ 12.1,  17.8,  18.9,  20.1]
estoques       = [  3.8,   4.2,   4.5,   4.6]
ativo_ncirc    = [ 47.9,  51.1,  52.8,  53.0]
imobilizado    = [ 28.1,  31.5,  33.4,  34.0]   # estimado
intangivel     = [ 12.3,  12.6,  11.4,  11.2]   # inclui goodwill/E2G
passivo_circ   = [ 16.1,  22.3,  28.4,  35.2]
passivo_ncirc  = [ 32.6,  38.4,  38.4,  46.5]
divida_bruta   = [ 18.4,  37.1,  52.0,  70.0]
pl             = [ 18.7,  14.5,  11.8,  -2.6]

def pct(num, den):
    return num / den * 100 if den != 0 else 0.0

print('=' * 78)
print('  ANÁLISE VERTICAL — DRE  (% da Receita Líquida)')
print('=' * 78)

dre_items = [
    ('Receita líquida',     receita),
    ('Lucro bruto',         lucro_bruto),
    ('EBITDA',              ebitda),
    ('EBIT',                ebit),
    ('Result. financeiro',  res_financeiro),
    ('Lucro líquido',       lucro_liq),
]

hdr = f"  {'Conta DRE':<26}" + ''.join(f'{p:>14}' for p in PERIODOS)
print(hdr)
print('  ' + '─' * (len(hdr) - 2))

for nome, serie in dre_items:
    vals = [pct(serie[i], receita[i]) for i in range(N)]
    linha = f"  {nome:<26}"
    for v in vals:
        sinal = '+' if v > 0 else ''
        cor_flag = ''
        if nome == 'Lucro líquido' and v < 0:
            cor_flag = ' ●'
        elif nome == 'EBIT' and v < 0:
            cor_flag = ' ●'
        elif nome == 'Result. financeiro' and abs(v) > 5:
            cor_flag = ' ▲'
        linha += f'{sinal}{v:>11.1f}%{cor_flag:<2}'
    print(linha)

print(f"""
  INTERPRETAÇÃO VERTICAL — DRE:
  • Margem bruta:   cai de 12,4% → 8,9%  (custo operacional crescendo + rápido que receita)
  • Margem EBITDA:  cai de  8,8% → 4,8%  (deterioração da geração operacional)
  • Result. financeiro: −1,7% → −7,2% da receita — ABSORVE toda geração operacional
  • Margem líquida: +1,3% (2021) → −8,8% (2024) — colapso de 10,1 p.p. em 4 anos
  ● = zona crítica / ▲ = pressão crescente
""")

print('=' * 78)
print('  ANÁLISE VERTICAL — BALANÇO PATRIMONIAL  (% do Ativo Total)')
print('=' * 78)

bp_items = [
    ('── ATIVO ──',          None),
    ('Ativo circulante',     ativo_circ),
    ('  ↳ Caixa e equiv.',   caixa),
    ('  ↳ Estoques',         estoques),
    ('Ativo não-circulante', ativo_ncirc),
    ('  ↳ Imobilizado',      imobilizado),
    ('  ↳ Intangível/E2G',   intangivel),
    ('── PASSIVO + PL ──',   None),
    ('Passivo circulante',   passivo_circ),
    ('Passivo não-circ.',    passivo_ncirc),
    ('  ↳ Dívida bruta',     divida_bruta),
    ('Patrimônio líquido',   pl),
]

hdr2 = f"  {'Conta Balanço':<26}" + ''.join(f'{p:>14}' for p in PERIODOS)
print(hdr2)
print('  ' + '─' * (len(hdr2) - 2))

for nome, serie in bp_items:
    if serie is None:
        print(f'  {nome}')
        continue
    vals = [pct(serie[i], ativo_total[i]) for i in range(N)]
    linha = f"  {nome:<26}"
    for i, v in enumerate(vals):
        flag = ''
        if nome == 'Patrimônio líquido' and serie[i] < 0:
            flag = ' ●'
        elif nome == '  ↳ Dívida bruta' and v > 60:
            flag = ' ●'
        sinal = '+' if v > 0 else ''
        linha += f'{sinal}{v:>11.1f}%{flag:<2}'
    print(linha)

print(f"""
  INTERPRETAÇÃO VERTICAL — BALANÇO:
  • Caixa/AT:   18,0% → 25,4% — aparente melhora, mas é dívida refinanciada
  • Dívida/AT:  27,3% → 88,5% — a empresa é quase inteiramente dívida em 2024/25
  • PL/AT:      27,8% → −3,3% — patrimônio desapareceu e tornou-se negativo ●
  • Passivo circ./AT: 23,9% → 44,5% — pressão de curto prazo extrema
  ● = zona crítica
""")


---
## 23. Índices de Liquidez

Medem a capacidade da empresa de honrar obrigações de curto e longo prazo com os ativos disponíveis.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ÍNDICES DE LIQUIDEZ
# Corrente · Seca · Imediata · Geral
# ══════════════════════════════════════════════════════════════════════════════

PERIODOS = ['2021/22', '2022/23', '2023/24', '2024/25']
N = 4

ativo_circ     = [19.5, 24.1, 25.8, 26.1]
estoques       = [ 3.8,  4.2,  4.5,  4.6]
caixa          = [12.1, 17.8, 18.9, 20.1]
passivo_circ   = [16.1, 22.3, 28.4, 35.2]
passivo_ncirc  = [32.6, 38.4, 38.4, 46.5]
ativo_ncirc    = [47.9, 51.1, 52.8, 53.0]

# Realizável L.P. (estimativa: ativo ncirc menos imobilizado/intangível)
realizavel_lp  = [ 7.5,  7.0,  5.0,  7.8]  # recebíveis LP, adiantamentos etc.

# ── Cálculo dos índices ───────────────────────────────────────────────────────
liq_corrente  = [ativo_circ[i] / passivo_circ[i] for i in range(N)]

liq_seca      = [(ativo_circ[i] - estoques[i]) / passivo_circ[i] for i in range(N)]

liq_imediata  = [caixa[i] / passivo_circ[i] for i in range(N)]

# Liquidez Geral = (AC + Realizável LP) / (PC + PNC)
passivo_total = [passivo_circ[i] + passivo_ncirc[i] for i in range(N)]
liq_geral     = [(ativo_circ[i] + realizavel_lp[i]) / passivo_total[i] for i in range(N)]

# Capital de Giro Líquido (CGL = AC - PC)
cgl = [ativo_circ[i] - passivo_circ[i] for i in range(N)]

# Prazo médio de recebimento estimado (dias) — proxy
prazo_rec = [round(((ativo_circ[i] - caixa[i] - estoques[i]) / (receita / 365)), 0)
             for i, receita in enumerate([161.0, 176.0, 184.0, 194.0])]

# Referências de mercado
REF = {
    'liq_corrente':  {'ok': 1.5, 'atencao': 1.0, 'critico': 0.9},
    'liq_seca':      {'ok': 1.0, 'atencao': 0.7, 'critico': 0.5},
    'liq_imediata':  {'ok': 0.5, 'atencao': 0.2, 'critico': 0.1},
    'liq_geral':     {'ok': 1.0, 'atencao': 0.7, 'critico': 0.5},
}

def semaforo(v, key):
    r = REF[key]
    if v >= r['ok']:     return 'ADEQUADO'
    if v >= r['atencao']:return 'ATENÇÃO '
    if v >= r['critico']:return 'ALERTA  '
    return 'CRÍTICO '

print('=' * 80)
print('  ÍNDICES DE LIQUIDEZ — RAÍZEN S.A.')
print('  Referências: liquidez corrente >1,5 (ótimo) | >1,0 (mínimo) | <1,0 (crítico)')
print('=' * 80)

print(f"\n  {'Índice':<28}  {'2021/22':>10}  {'2022/23':>10}  {'2023/24':>10}  {'2024/25':>10}  {'Limiar OK':>10}")
print('  ' + '─' * 82)

indices = [
    ('Liq. Corrente   (AC/PC)',     liq_corrente,  'liq_corrente',  1.5),
    ('Liq. Seca (AC-Est)/PC',       liq_seca,      'liq_seca',      1.0),
    ('Liq. Imediata (Caixa/PC)',    liq_imediata,  'liq_imediata',  0.5),
    ('Liq. Geral  (AC+RLP)/(PC+PNC)', liq_geral,  'liq_geral',     1.0),
]

for nome, serie, key, ref_ok in indices:
    linha = f"  {nome:<28}"
    for i, v in enumerate(serie):
        status = semaforo(v, key)
        linha += f'  {v:>6.3f}×'
    linha += f'  {ref_ok:>9}×'
    print(linha)

print()
print('  SEMAFORIZAÇÃO (ADEQUADO / ATENÇÃO / ALERTA / CRÍTICO):')
print(f"  {'Índice':<28}  " + '  '.join(f'{p:>9}' for p in PERIODOS))
print('  ' + '─' * 68)

for nome, serie, key, ref_ok in indices:
    linha = f"  {nome:<28}"
    for i, v in enumerate(serie):
        s = semaforo(v, key)
        linha += f'  {s:>9}'
    print(linha)

print(f"""
  CAPITAL DE GIRO LÍQUIDO (CGL = AC − PC) — em R$ bilhões:
  {'Período':<12}  {'CGL R$ bi':>11}  {'Situação':>12}
  {'─'*38}""")

for i in range(N):
    sit = 'Positivo ●' if cgl[i] > 0 else 'NEGATIVO ●'
    mark = '' if cgl[i] > 0 else '  ← CRÍTICO'
    print(f"  {PERIODOS[i]:<12}  {cgl[i]:>+10.1f}  {sit:>12}{mark}")

print(f"""
  DIAGNÓSTICO:
  • 2021/22–2022/23: CGL positivo mas encolhendo rapidamente
  • 2023/24:  CGL torna-se NEGATIVO (−R$2,6 bi) — empresa precisa de dívida
              para financiar o giro: passivo CP > ativo CP
  • 2024/25:  CGL = −R$9,1 bi — insolvência operacional de curto prazo
              Cada real de caixa é lastreado por R$1,35 de dívida CP

  LIQUIDEZ IMEDIATA PARADOXO:
  O caixa cresce (R$12,1→R$20,1 bi) mas a liquidez imediata CAI (0,75→0,57×)
  → Passivo circulante cresceu 2,19× mais rápido que o caixa
  → O caixa é em grande parte vinculado / garantia de dívidas reestruturadas
""")


---
## 24. Índices de Endividamento

Avaliam o nível e a qualidade da dívida — quanto da estrutura de capital é financiada por terceiros e em que prazo.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ÍNDICES DE ENDIVIDAMENTO
# Endividamento geral · Composição do endividamento · Imobilização do PL
# Dívida líquida/EBITDA · Cobertura de juros · Grau de alavancagem financeira
# ══════════════════════════════════════════════════════════════════════════════

PERIODOS = ['2021/22', '2022/23', '2023/24', '2024/25']
N = 4

receita        = [161.0, 176.0, 184.0, 194.0]
ebitda         = [ 14.2,  13.8,  12.1,   9.4]
ebit           = [  5.2,   3.7,  -2.6,  -9.9]
lucro_liq      = [  2.1,   1.4,  -3.1, -17.2]
fco            = [  7.2,   8.6,   6.4,   5.8]
juros_pagos    = [  3.4,   5.8,   9.2,  12.1]
ativo_total    = [ 67.4,  75.2,  78.6,  79.1]
ativo_circ     = [ 19.5,  24.1,  25.8,  26.1]
ativo_ncirc    = [ 47.9,  51.1,  52.8,  53.0]
imobilizado    = [ 28.1,  31.5,  33.4,  34.0]
passivo_circ   = [ 16.1,  22.3,  28.4,  35.2]
passivo_ncirc  = [ 32.6,  38.4,  38.4,  46.5]
divida_bruta   = [ 18.4,  37.1,  52.0,  70.0]
caixa          = [ 12.1,  17.8,  18.9,  20.1]
pl             = [ 18.7,  14.5,  11.8,  -2.6]

passivo_total  = [passivo_circ[i] + passivo_ncirc[i] for i in range(N)]
divida_liq     = [divida_bruta[i] - caixa[i] for i in range(N)]

# ── Índices ───────────────────────────────────────────────────────────────────
# 1. Endividamento Geral = Passivo Total / Ativo Total (quanto do ativo é dívida)
end_geral = [passivo_total[i] / ativo_total[i] * 100 for i in range(N)]

# 2. Composição do Endividamento = PC / (PC + PNC) — % da dívida no curto prazo
comp_end  = [passivo_circ[i] / passivo_total[i] * 100 for i in range(N)]

# 3. Imobilização do PL = Ativo não-circ / PL — quanto do PL está imobilizado
imob_pl   = [ativo_ncirc[i] / pl[i] if pl[i] > 0 else float('inf') for i in range(N)]

# 4. Participação de Capitais de Terceiros = Passivo Total / PL
pct_terceiros = [passivo_total[i] / pl[i] if pl[i] > 0 else float('inf') for i in range(N)]

# 5. Dívida Líq. / EBITDA — Alavancagem do credor
dl_ebitda = [divida_liq[i] / ebitda[i] if ebitda[i] > 0 else float('inf') for i in range(N)]

# 6. Cobertura de juros pelo EBIT (EBIT / Desp. financeira)
cob_ebit  = [ebit[i] / abs(juros_pagos[i]) if juros_pagos[i] != 0 else 0 for i in range(N)]

# 7. Cobertura de juros pelo FCO
cob_fco   = [fco[i] / juros_pagos[i] for i in range(N)]

# 8. Grau de Alavancagem Financeira (GAF) = EBIT / (EBIT − Juros)
# GAF > 1: endividamento amplifica variações do resultado
gaf = []
for i in range(N):
    lajir = ebit[i]
    lair  = ebit[i] - juros_pagos[i]
    if lair != 0 and lajir != 0:
        gaf.append(abs(lajir / lair))
    else:
        gaf.append(None)

# 9. Endividamento Oneroso = Dívida Bruta / AT
end_oneroso = [divida_bruta[i] / ativo_total[i] * 100 for i in range(N)]

print('=' * 80)
print('  ÍNDICES DE ENDIVIDAMENTO — RAÍZEN S.A.')
print('  Fonte: CVM/DFP consolidado · Cálculos próprios')
print('=' * 80)

print(f"\n  {'Índice':<36}  {'2021/22':>9}  {'2022/23':>9}  {'2023/24':>9}  {'2024/25':>9}  {'Ref. OK':>9}")
print('  ' + '─' * 84)

def fmt_num(v, suf='', dec=2, inf_str='∞ (neg.PL)'):
    if v is None: return '   N/D   '
    if v == float('inf'): return f' {inf_str}'
    return f'{v:>+{dec+5}.{dec}f}{suf}'

rows = [
    ('Endividamento Geral (%)',         end_geral,   '%',  '<50%'),
    ('Composição Endiv. (CP%) ',        comp_end,    '%',  '<40%'),
    ('Part. Cap. Terceiros (PT/PL)',     pct_terceiros,'×', '<2,0×'),
    ('Imobilização do PL',              imob_pl,     '×',  '<1,5×'),
    ('End. Oneroso (Dívida/AT)',         end_oneroso, '%',  '<30%'),
    ('Dívida Líq. / EBITDA',            dl_ebitda,   '×',  '<2,5×'),
    ('Cobertura Juros EBIT (×)',         cob_ebit,    '×',  '>2,0×'),
    ('Cobertura Juros FCO (×)',          cob_fco,     '×',  '>1,5×'),
]

for nome, serie, suf, ref in rows:
    linha = f"  {nome:<36}"
    for v in serie:
        if v == float('inf') or v is None:
            linha += f'  {"∞/neg":>9}'
        else:
            linha += f'  {v:>+9.2f}'
    linha += f'  {ref:>9}'
    print(linha)

print()
print('  GRAU DE ALAVANCAGEM FINANCEIRA (GAF):')
for i in range(N):
    v = gaf[i]
    if v is None:
        print(f'  {PERIODOS[i]}: GAF = indefinido (EBIT ou LAIR = 0)')
    elif v > 3:
        print(f'  {PERIODOS[i]}: GAF = {v:.2f}×  ← ALAVANCAGEM EXTREMA: cada 1% de var. no EBIT → {v:.1f}% no LL')
    else:
        print(f'  {PERIODOS[i]}: GAF = {v:.2f}×')

print(f"""
  ANÁLISE DE QUALIDADE DA DÍVIDA:
  {'Período':<12}  {'Dívida bruta':>14}  {'Dívida liq.':>12}  {'DL/EBITDA':>10}  {'% Curto Prazo':>14}
  {'─'*66}""")
for i in range(N):
    dl_eb = f'{dl_ebitda[i]:.1f}×' if dl_ebitda[i] != float('inf') else '∞'
    print(f"  {PERIODOS[i]:<12}  R$ {divida_bruta[i]:>9.1f} bi  R$ {divida_liq[i]:>8.1f} bi  {dl_eb:>10}  {comp_end[i]:>13.1f}%")

print(f"""
  DIAGNÓSTICO DE ENDIVIDAMENTO:
  ✦ 2021/22: Estrutura agressiva mas gerenciável — DL/EBITDA 0,4× (dívida quase zerada)
  ✦ 2022/23: INFLEXÃO — DL/EBITDA salta para 1,4× · Participação de terceiros: 4,2×
  ✦ 2023/24: Zona de risco — composição CP cresce de 37% para 43% da dívida total
  ✦ 2024/25: PL negativo → índices de imobilização e Part. CP3 tornam-se inválidos
              End. oneroso: {end_oneroso[3]:.1f}% do AT — a empresa é basicamente dívida
              DL/EBITDA: {dl_ebitda[3]:.1f}× — limite de crédito corporativo é tipicamente 3,5–4,5×
              Cobertura EBIT: {cob_ebit[3]:.2f}× (negativa) — EBIT não paga mais nem parte dos juros
""")


---
## 25. Índices de Solvência

Avaliam a capacidade estrutural de longo prazo de honrar todas as obrigações — uma empresa insolvente pode estar operando mas tem passivo > ativo.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ÍNDICES DE SOLVÊNCIA
# Solvência geral · Grau de insolvência · Ponto de equilíbrio financeiro
# Capacidade de geração de caixa vs. serviço da dívida
# ══════════════════════════════════════════════════════════════════════════════
import math

PERIODOS = ['2021/22', '2022/23', '2023/24', '2024/25']
N = 4

ativo_total    = [67.4, 75.2, 78.6, 79.1]
ativo_circ     = [19.5, 24.1, 25.8, 26.1]
realizavel_lp  = [ 7.5,  7.0,  5.0,  7.8]
passivo_circ   = [16.1, 22.3, 28.4, 35.2]
passivo_ncirc  = [32.6, 38.4, 38.4, 46.5]
divida_bruta   = [18.4, 37.1, 52.0, 70.0]
caixa          = [12.1, 17.8, 18.9, 20.1]
pl             = [18.7, 14.5, 11.8, -2.6]
ebitda         = [14.2, 13.8, 12.1,  9.4]
ebit           = [ 5.2,  3.7, -2.6, -9.9]
lucro_liq      = [ 2.1,  1.4, -3.1,-17.2]
fco            = [ 7.2,  8.6,  6.4,  5.8]
capex          = [10.8, 14.1, 13.1, 14.2]
juros_pagos    = [ 3.4,  5.8,  9.2, 12.1]
receita        = [161.0,176.0,184.0,194.0]
vm_pl          = [29.2, 18.2,  9.8,  3.7]   # valor de mercado do PL

passivo_total  = [passivo_circ[i] + passivo_ncirc[i] for i in range(N)]
divida_liq     = [divida_bruta[i] - caixa[i] for i in range(N)]
fcl            = [fco[i] - capex[i] for i in range(N)]

# ── 1. SOLVÊNCIA GERAL = (AC + Realizável LP) / (PC + PNC) ───────────────────
solv_geral = [(ativo_circ[i] + realizavel_lp[i]) / passivo_total[i] for i in range(N)]

# ── 2. ÍNDICE DE INSOLVÊNCIA = Passivo Total / Ativo Total ────────────────────
# >1 significa passivo supera ativo (insolvência técnica)
idx_insolv = [passivo_total[i] / ativo_total[i] for i in range(N)]

# ── 3. MARGEM DE SOLVÊNCIA = (AT - PT) = PL ──────────────────────────────────
margem_solv = pl  # equivalente

# ── 4. COBERTURA DO SERVIÇO DA DÍVIDA (DSCR) ─────────────────────────────────
# DSCR = FCO / (Juros + Amortizações no período)
# Amortizações estimadas como 15% da dívida bruta (benchmark setor)
amort_est   = [divida_bruta[i] * 0.15 for i in range(N)]
dscr        = [fco[i] / (juros_pagos[i] + amort_est[i]) for i in range(N)]

# ── 5. PRAZO DE EXTINÇÃO DA DÍVIDA ───────────────────────────────────────────
# Anos necessários para quitar dívida líquida com FCO atual (sem crescimento)
prazo_extincao = []
for i in range(N):
    if fco[i] > 0 and divida_liq[i] > 0:
        prazo_extincao.append(divida_liq[i] / fco[i])
    else:
        prazo_extincao.append(None)

# ── 6. ÍNDICE DE SUSTENTABILIDADE DO CAPEX ───────────────────────────────────
# FCO / Capex — quanto do capex é autofinanciado
sust_capex = [fco[i] / capex[i] for i in range(N)]

# ── 7. RETORNO SOBRE ATIVO (ROA) e RETORNO SOBRE PL (ROE) ────────────────────
roa = [lucro_liq[i] / ativo_total[i] * 100 for i in range(N)]
roe = [lucro_liq[i] / pl[i] * 100 if pl[i] != 0 else None for i in range(N)]

# ── 8. ALTMAN Z (reapresentado no contexto de solvência) ─────────────────────
def altman_z(i):
    cg  = ativo_circ[i] - passivo_circ[i]
    at  = ativo_total[i]
    pt  = passivo_total[i]
    X1  = cg / at
    X2  = pl[i] * 0.6 / at
    X3  = ebit[i] / at
    X4  = vm_pl[i] / pt
    X5  = receita[i] / at
    Z   = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
    return Z

altmans = [altman_z(i) for i in range(N)]

print('=' * 80)
print('  ÍNDICES DE SOLVÊNCIA — RAÍZEN S.A.')
print('  Solvência mede a capacidade ESTRUTURAL de longo prazo de honrar obrigações')
print('=' * 80)

print(f"\n  {'Índice':<38}  {'2021/22':>9}  {'2022/23':>9}  {'2023/24':>9}  {'2024/25':>9}  Ref.")
print('  ' + '─' * 84)

def fmt(v, dec=3, suf='', na='  N/D   '):
    if v is None: return na
    if v == float('inf'): return '  ∞      '
    return f'{v:>+10.{dec}f}{suf}'

rows = [
    ('Solvência Geral ((AC+RLP)/(PC+PNC))', solv_geral,      '>1,0×'),
    ('Índice Insolvência (PT/AT)',          idx_insolv,       '<0,7×'),
    ('Margem de Solvência (PL) R$ bi',      margem_solv,      '>0'),
    ('DSCR (FCO/(Juros+Amort.))',           dscr,             '>1,2×'),
    ('Sustentabilidade Capex (FCO/Capex)',  sust_capex,       '>1,0×'),
    ('ROA — Retorno s/ Ativo (%)',          roa,              '>5%'),
    ('ROE — Retorno s/ PL (%)',             roe,              '>10%'),
    ('Altman Z-Score',                      altmans,          '>2,99'),
]

for nome, serie, ref in rows:
    linha = f"  {nome:<38}"
    for v in serie:
        if v is None:
            linha += f'  {"∞/neg":>9}'
        else:
            linha += f'  {v:>+9.3f}'
    linha += f'  {ref}'
    print(linha)

print(f"""
  PRAZO ESTIMADO DE EXTINÇÃO DA DÍVIDA LÍQUIDA (anos, à FCO constante):
  {'Período':<12}  {'Dívida líq.':>12}  {'FCO':>8}  {'Prazo (anos)':>13}  {'Diagnóstico':>20}""")

diag_prazo = ['Gerenciável (5a)', 'Alongado (6a)', 'Crítico (11a)', 'Impossível — RE']
for i in range(N):
    p = prazo_extincao[i]
    p_str = f'{p:.1f} anos' if p else 'N/D'
    print(f"  {PERIODOS[i]:<12}  R$ {divida_liq[i]:>8.1f} bi  R$ {fco[i]:>4.1f} bi  {p_str:>13}  {diag_prazo[i]:>20}")

print(f"""
  QUADRO RESUMO DE SOLVÊNCIA:
  ┌─────────────────────────────────────────────────────────────────────┐
  │  Dimensão              2021/22     2022/23     2023/24     2024/25  │
  ├─────────────────────────────────────────────────────────────────────┤
  │  Solvência geral          {solv_geral[0]:.2f}×       {solv_geral[1]:.2f}×       {solv_geral[2]:.2f}×      {solv_geral[3]:.2f}×  │
  │  Insolvência (PT/AT)      {idx_insolv[0]:.2f}×       {idx_insolv[1]:.2f}×       {idx_insolv[2]:.2f}×      {idx_insolv[3]:.2f}×  │
  │  DSCR (cobertura)         {dscr[0]:.2f}×       {dscr[1]:.2f}×       {dscr[2]:.2f}×      {dscr[3]:.2f}×  │
  │  Altman Z-Score           {altmans[0]:.2f}        {altmans[1]:.2f}        {altmans[2]:.2f}       {altmans[3]:.2f}  │
  │  Zona Altman            Cinza      Perigo      Perigo      Colapso  │
  └─────────────────────────────────────────────────────────────────────┘

  MARCOS DE SOLVÊNCIA:
  ✦ 2021/22: Solvente — Altman Z 2,18 (zona cinza), mas DSCR já abaixo de 1,2
  ✦ 2022/23: Insolvência a caminho — Altman zona perigo · DL/EBITDA 1,4×
  ✦ 2023/24: Insolvência operacional — DSCR 0,43× · PL encolhendo aceleradamente
  ✦ 2024/25: Insolvência formal — PT/AT = {idx_insolv[3]:.2f}× · PL NEGATIVO · Altman {altmans[3]:.2f}
              A empresa só sobreviveu por captações de R$27,8 bi em 2024/25
              Sem acesso ao mercado de dívida → RE era inevitável
""")


DASHBOARD

In [8]:
"""
Raízen S.A. — Gerador do Dashboard Completo
============================================
Gera o arquivo raizen_dashboard_completo.html com análise completa:
  - Financeiro (DRE, DFC, Balanço)
  - Estatístico (Z-Score, Altman, IQR, Score de risco)
  - Projeções (OLS, IC 95%, Cenários)
  - Macro (Mercado de etanol, Guerra de custos)
  - Concorrentes (Raízen vs Inpasa vs FS)
  - Red Flags (financeiros + setoriais)
  - Cronologia
  - Resumo integrado

Dependências: Python 3.8+ — NENHUMA biblioteca externa necessária.
  - math, json (stdlib)
  - Gráficos via Chart.js carregado pelo navegador via CDN

Uso:
  python raizen_gerar_dashboard.py
  # Abre raizen_dashboard_completo.html no navegador
"""

import math
import json
import os
import webbrowser

# ══════════════════════════════════════════════════════════════════════════════
# 1. DADOS HISTÓRICOS — CVM / DFP consolidado
#    Valores em R$ bilhões · Ano fiscal: abril → março
# ══════════════════════════════════════════════════════════════════════════════

PERIODOS = ['2021/22', '2022/23', '2023/24', '2024/25']
N = len(PERIODOS)

# DRE
receita        = [161.0, 176.0, 184.0, 194.0]
lucro_bruto    = [ 20.0,  20.8,  18.8,  17.3]
ebit           = [  5.2,   3.7,  -2.6,  -9.9]
res_financeiro = [ -2.8,  -5.2,  -9.1, -14.0]
lucro_liq      = [  2.1,   1.4,  -3.1, -17.2]
ebitda         = [ 14.2,  13.8,  12.1,   9.4]

# Balanço
ativo_total    = [ 67.4,  75.2,  78.6,  79.1]
ativo_circ     = [ 19.5,  24.1,  25.8,  26.1]
caixa          = [ 12.1,  17.8,  18.9,  20.1]
estoques       = [  3.8,   4.2,   4.5,   4.6]
passivo_circ   = [ 16.1,  22.3,  28.4,  35.2]
passivo_ncirc  = [ 32.6,  38.4,  38.4,  46.5]
pl             = [ 18.7,  14.5,  11.8,  -2.6]
divida_bruta   = [ 18.4,  37.1,  52.0,  70.0]

# DFC
fco            = [  7.2,   8.6,   6.4,   5.8]
capex          = [ 10.8,  14.1,  13.1,  14.2]
juros_pagos    = [  3.4,   5.8,   9.2,  12.1]
captacoes      = [ 18.2,  24.1,  26.3,  27.8]

# Mercado (para Altman X4)
vm_pl          = [ 29.2,  18.2,   9.8,   3.7]

# Derivados
passivo_total  = [passivo_circ[i] + passivo_ncirc[i] for i in range(N)]
fcl            = [fco[i] - capex[i]                  for i in range(N)]
capital_giro   = [ativo_circ[i] - passivo_circ[i]    for i in range(N)]
lucros_ret     = [pl[i] * 0.6                        for i in range(N)]

# Indicadores financeiros calculados
liq_cor  = [ativo_circ[i] / passivo_circ[i]                                 for i in range(N)]
liq_seca = [(ativo_circ[i] - estoques[i]) / passivo_circ[i]                 for i in range(N)]
alav     = [passivo_total[i] / pl[i] if pl[i] != 0 else 99.0                for i in range(N)]
end_ger  = [passivo_total[i] / ativo_total[i] * 100                         for i in range(N)]
mg_bruta = [lucro_bruto[i] / receita[i] * 100                               for i in range(N)]
mg_ebit  = [ebit[i] / receita[i] * 100                                      for i in range(N)]
mg_liq   = [lucro_liq[i] / receita[i] * 100                                 for i in range(N)]
roa      = [lucro_liq[i] / ativo_total[i] * 100                             for i in range(N)]
cobert_j = [fco[i] / juros_pagos[i]                                         for i in range(N)]
cap_fco  = [capex[i] / fco[i]                                                for i in range(N)]
giro_at  = [receita[i] / ativo_total[i]                                     for i in range(N)]

# ══════════════════════════════════════════════════════════════════════════════
# 2. FUNÇÕES ESTATÍSTICAS — Python puro, sem bibliotecas
# ══════════════════════════════════════════════════════════════════════════════

def media(lst):
    return sum(lst) / len(lst)

def desvio_padrao(lst, amostral=True):
    """Desvio-padrão amostral (n-1, correção de Bessel) ou populacional."""
    m, n = media(lst), len(lst)
    divisor = (n - 1) if amostral else n
    return math.sqrt(sum((x - m) ** 2 for x in lst) / divisor)

def percentil(lst_ord, p):
    """Percentil com interpolação linear — sem bibliotecas."""
    idx = (p / 100) * (len(lst_ord) - 1)
    lo  = int(idx)
    hi  = min(lo + 1, len(lst_ord) - 1)
    return lst_ord[lo] + (idx - lo) * (lst_ord[hi] - lst_ord[lo])

def z_scores(lst):
    """Z = (x - μ) / σ · Aviso: n=4 é indicador direcional, não teste formal."""
    m, s = media(lst), desvio_padrao(lst)
    return [(x - m) / s if s else 0.0 for x in lst]

def iqr_limites(lst):
    """Limites IQR = P25 ± 1.5×IQR."""
    s = sorted(lst)
    p25 = percentil(s, 25)
    p75 = percentil(s, 75)
    iqr = p75 - p25
    return p25 - 1.5 * iqr, p75 + 1.5 * iqr

def ols(y):
    """
    Regressão linear OLS (Mínimos Quadrados Ordinários) — Python puro.
    b = (n×Σxy − Σx×Σy) / (n×Σx² − (Σx)²)
    a = ȳ − b×x̄
    R² = 1 − SS_res / SS_tot
    RMSE = √(SS_res / n)
    """
    n = len(y)
    x = list(range(n))
    sx  = sum(x)
    sy  = sum(y)
    sxy = sum(x[i] * y[i] for i in range(n))
    sx2 = sum(xi ** 2 for xi in x)
    den = n * sx2 - sx ** 2
    if not den:
        return None
    b   = (n * sxy - sx * sy) / den
    a   = (sy - b * sx) / n
    yh  = [a + b * xi for xi in x]
    my  = sy / n
    sst = sum((yi - my) ** 2 for yi in y)
    ssr = sum((y[i] - yh[i]) ** 2 for i in range(n))
    r2  = 1 - ssr / sst if sst else 0
    rmse = math.sqrt(ssr / n)
    return {'a': a, 'b': b, 'r2': r2, 'rmse': rmse,
            'yh': yh, 'res': [y[i] - yh[i] for i in range(n)]}

def projetar(reg, n_ahead):
    """Projeta n_ahead períodos · IC 95% ≈ ±1.96×RMSE."""
    return [{
        'yp':  reg['a'] + reg['b'] * (N + k),
        'lo': (reg['a'] + reg['b'] * (N + k)) - 1.96 * reg['rmse'],
        'hi': (reg['a'] + reg['b'] * (N + k)) + 1.96 * reg['rmse'],
    } for k in range(n_ahead)]

def altman_z(i):
    """
    Altman Z-Score (1968) — empresas industriais listadas.
    Z = 1.2×X1 + 1.4×X2 + 3.3×X3 + 0.6×X4 + 1.0×X5
    Zonas: Z>2.99 Segura · 1.81–2.99 Cinza · <1.81 Perigo
    """
    cg = ativo_circ[i] - passivo_circ[i]
    at = ativo_total[i]
    lr = lucros_ret[i]
    pt = passivo_total[i]
    X1 = cg / at
    X2 = lr / at
    X3 = ebit[i] / at
    X4 = vm_pl[i] / pt
    X5 = receita[i] / at
    Z  = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
    zona = 'Segura' if Z > 2.99 else ('Cinza' if Z >= 1.81 else 'Perigo')
    return {'X1': X1, 'X2': X2, 'X3': X3, 'X4': X4, 'X5': X5, 'Z': Z, 'zona': zona}

def norm_0_100(v, lo, hi, inv=False):
    """Normaliza para 0–100 com referências explícitas."""
    n = max(0, min(100, (v - lo) / (hi - lo) * 100))
    return n if inv else 100 - n

def score_risco(i):
    """Score de risco composto — 5 dimensões ponderadas, 0=saudável, 100=colapso."""
    az = altman_z(i)['Z']
    sL = norm_0_100(liq_cor[i], 0.5, 2.0, False) * 0.6 + norm_0_100(liq_seca[i], 0.4, 1.5, False) * 0.4
    al = min(alav[i] if alav[i] != 99.0 else 10, 10)
    sE = norm_0_100(al, 1.0, 6.0, True)  * 0.6 + norm_0_100(end_ger[i], 40, 95, True) * 0.4
    sR = norm_0_100(mg_liq[i], -15, 8, False) * 0.5 + norm_0_100(mg_ebit[i], -8, 6, False) * 0.5
    sC = norm_0_100(cobert_j[i], 0.0, 2.5, False) * 0.40 + \
         norm_0_100(fcl[i], -10, 2.0, False)       * 0.35 + \
         norm_0_100(cap_fco[i], 0.5, 3.0, True)    * 0.25
    sS = norm_0_100(pl[i], -5, 25, False) * 0.5 + norm_0_100(az, -1, 3.5, False) * 0.5
    saude = sL*0.20 + sE*0.25 + sR*0.20 + sC*0.25 + sS*0.10
    return {
        'liq': min(100 - sL + 5, 100),
        'end': min(100 - sE + 5, 100),
        'ren': min(100 - sR + 5, 100),
        'cai': min(100 - sC + 5, 100),
        'sol': min(100 - sS + 5, 100),
        'comp': min(100 - saude + 15, 100),
    }

# ══════════════════════════════════════════════════════════════════════════════
# 3. PRÉ-CALCULAR TUDO EM PYTHON — serializar para JSON → injetar no HTML
# ══════════════════════════════════════════════════════════════════════════════

def r2(v, dec=3):
    """Arredonda com segurança."""
    return round(float(v), dec)

altmans  = [altman_z(i) for i in range(N)]
scores   = [score_risco(i) for i in range(N)]
ols_dv   = ols(divida_bruta)
ols_ll   = ols(lucro_liq)
ols_fco  = ols(fco)
ols_cob  = ols(cobert_j)
ols_alav = ols(alav)
ols_ml   = ols(mg_liq)

proj_dv  = projetar(ols_dv,  2)
proj_ll  = projetar(ols_ll,  2)
proj_cob = projetar(ols_cob, 2)

zs_ll   = z_scores(lucro_liq)
zs_dv   = z_scores(divida_bruta)
zs_jr   = z_scores(juros_pagos)
zs_cob  = z_scores(cobert_j)
zs_fco  = z_scores(fco)
zs_alav = z_scores(alav)

iqr_lo_dv, iqr_hi_dv = iqr_limites(divida_bruta)
iqr_lo_cb, iqr_hi_cb = iqr_limites(cobert_j)

# Deterioração
detVars = [
    ('Lucro liq.', lucro_liq),
    ('EBIT',        ebit),
    ('FCO',         fco),
    ('Juros',       juros_pagos),
    ('Dívida',      divida_bruta),
    ('Marg.liq%',   mg_liq),
    ('Cobert.',     cobert_j),
]
det_mat = []
for nome, serie in detVars:
    row = []
    for k in range(1, N):
        if serie[k-1] != 0:
            row.append(round((serie[k] - serie[k-1]) / abs(serie[k-1]) * 100, 1))
        else:
            row.append(0)
    det_mat.append({'nome': nome, 'vals': row})

delta   = [round(lucro_liq[i] - lucro_liq[i-1], 2) for i in range(1, N)]
delta2  = [round(delta[i] - delta[i-1], 2) for i in range(1, len(delta))]

# Macro
SAFRAS_M = ['2019/20','2020/21','2021/22','2022/23','2023/24','2024/25']
prod_cana  = [29.8, 29.1, 29.3, 29.8, 30.2, 30.5]
prod_milho = [ 2.0,  2.6,  4.0,  6.0,  7.8, 10.2]
share_milho = [round(m/(m+c)*100, 1) for m, c in zip(prod_milho, prod_cana)]
custo_milho = [1.52, 1.60, 1.72, 1.95, 2.10, 1.88]
custo_cana  = [1.70, 1.85, 2.00, 2.15, 2.22, 2.36]

# Cenários
def simular_cenario(ddiv, dfco, selic, steps=2):
    res = [{'d': 70.0, 'f': 5.8, 'cob': round(5.8 / (70.0 * selic), 3)}]
    for _ in range(steps):
        d = round(res[-1]['d'] * (1 + ddiv), 2)
        f = round(res[-1]['f'] * (1 + dfco), 2)
        res.append({'d': d, 'f': f, 'cob': round(f / (d * selic), 3)})
    return res

cen_base = simular_cenario(+0.25, -0.10, 0.15)
cen_recu = simular_cenario(-0.35, +0.10, 0.12)
cen_coll = simular_cenario(+0.40, -0.25, 0.15)

# Serializar tudo para JSON
dados_json = json.dumps({
    'AN': PERIODOS,
    'ANp': PERIODOS + ['2025/26p', '2026/27p'],
    'receita': receita, 'lucro_liq': lucro_liq, 'ebit': ebit,
    'ebitda': ebitda, 'res_financeiro': res_financeiro,
    'fco': fco, 'capex': capex, 'juros_pagos': juros_pagos,
    'fcl': fcl, 'divida_bruta': divida_bruta, 'pl': pl,
    'ativo_circ': ativo_circ, 'passivo_circ': passivo_circ,
    'ativo_total': ativo_total,
    'liq_cor': [r2(v) for v in liq_cor],
    'liq_seca': [r2(v) for v in liq_seca],
    'alav': [r2(v) for v in alav],
    'end_ger': [r2(v) for v in end_ger],
    'mg_liq': [r2(v) for v in mg_liq],
    'mg_ebit': [r2(v) for v in mg_ebit],
    'mg_bruta': [r2(v) for v in mg_bruta],
    'roa': [r2(v) for v in roa],
    'cobert_j': [r2(v) for v in cobert_j],
    'cap_fco': [r2(v) for v in cap_fco],
    # Z-Scores
    'zs_ll': [r2(v) for v in zs_ll],
    'zs_dv': [r2(v) for v in zs_dv],
    'zs_jr': [r2(v) for v in zs_jr],
    'zs_cob': [r2(v) for v in zs_cob],
    'zs_fco': [r2(v) for v in zs_fco],
    'zs_alav': [r2(v) for v in zs_alav],
    # IQR
    'iqr_hi_dv': r2(iqr_hi_dv), 'iqr_lo_dv': r2(iqr_lo_dv),
    'iqr_hi_cb': r2(iqr_hi_cb), 'iqr_lo_cb': r2(iqr_lo_cb),
    # Altman
    'altmans': [{'Z': r2(a['Z'],3), 'X1': r2(a['X1'],4), 'X2': r2(a['X2'],4),
                 'X3': r2(a['X3'],4), 'X4': r2(a['X4'],4), 'X5': r2(a['X5'],4),
                 'zona': a['zona']} for a in altmans],
    # OLS
    'ols_dv': {'a': r2(ols_dv['a']), 'b': r2(ols_dv['b']), 'r2': r2(ols_dv['r2'],4), 'rmse': r2(ols_dv['rmse']), 'yh': [r2(v) for v in ols_dv['yh']], 'res': [r2(v) for v in ols_dv['res']]},
    'ols_ll': {'a': r2(ols_ll['a']), 'b': r2(ols_ll['b']), 'r2': r2(ols_ll['r2'],4), 'rmse': r2(ols_ll['rmse']), 'yh': [r2(v) for v in ols_ll['yh']], 'res': [r2(v) for v in ols_ll['res']]},
    'ols_fco': {'a': r2(ols_fco['a']), 'b': r2(ols_fco['b']), 'r2': r2(ols_fco['r2'],4), 'rmse': r2(ols_fco['rmse'])},
    'ols_cob': {'a': r2(ols_cob['a']), 'b': r2(ols_cob['b']), 'r2': r2(ols_cob['r2'],4), 'rmse': r2(ols_cob['rmse'])},
    'ols_alav': {'a': r2(ols_alav['a']), 'b': r2(ols_alav['b']), 'r2': r2(ols_alav['r2'],4), 'rmse': r2(ols_alav['rmse'])},
    'ols_ml': {'a': r2(ols_ml['a']), 'b': r2(ols_ml['b']), 'r2': r2(ols_ml['r2'],4), 'rmse': r2(ols_ml['rmse'])},
    # Projeções
    'proj_dv': [{'yp': r2(p['yp']), 'lo': r2(p['lo']), 'hi': r2(p['hi'])} for p in proj_dv],
    'proj_ll': [{'yp': r2(p['yp']), 'lo': r2(p['lo']), 'hi': r2(p['hi'])} for p in proj_ll],
    'proj_cob': [{'yp': r2(p['yp']), 'lo': r2(p['lo']), 'hi': r2(p['hi'])} for p in proj_cob],
    # Scores
    'scores': [{'liq': r2(s['liq']), 'end': r2(s['end']), 'ren': r2(s['ren']),
                'cai': r2(s['cai']), 'sol': r2(s['sol']), 'comp': r2(s['comp'])} for s in scores],
    # Deterioração
    'det_mat': det_mat, 'det_nomes': [d['nome'] for d in det_mat],
    'delta': delta, 'delta2': delta2,
    # Macro
    'safras': SAFRAS_M, 'prod_cana': prod_cana, 'prod_milho': prod_milho,
    'share_milho': share_milho, 'custo_milho': custo_milho, 'custo_cana': custo_cana,
    # Cenários
    'cen_base': cen_base, 'cen_recu': cen_recu, 'cen_coll': cen_coll,
}, ensure_ascii=False, indent=2)

print("Dados calculados em Python puro:")
print(f"  Altman Z 2024/25 = {altmans[3]['Z']:.4f} ({altmans[3]['zona']})")
print(f"  Score risco 2024/25 = {scores[3]['comp']:.1f}/100")
print(f"  OLS dívida: b={ols_dv['b']:.3f} bi/período, R²={ols_dv['r2']:.4f}")
print(f"  Proj. dívida 2025/26 = R$ {proj_dv[0]['yp']:.1f} bi [IC: {proj_dv[0]['lo']:.1f}–{proj_dv[0]['hi']:.1f}]")
print(f"  Z-Score LL 2024/25 = {zs_ll[3]:.3f}σ")
print(f"  Limite IQR superior dívida = R$ {iqr_hi_dv:.1f} bi")

# ══════════════════════════════════════════════════════════════════════════════
# 4. TEMPLATE HTML — os dados vêm do Python via JSON, gráficos via Chart.js
# ══════════════════════════════════════════════════════════════════════════════

HTML = r"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>Raízen S.A. — Análise Completa</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;background:#f5f4f0;color:#1a1a18;font-size:13px}
.nav{position:sticky;top:0;z-index:99;background:#fff;border-bottom:1px solid #e0dfd8;display:flex;align-items:center;padding:0 20px;height:48px;gap:4px;overflow-x:auto}
.nav::-webkit-scrollbar{display:none}
.brand{font-size:12px;font-weight:600;padding-right:16px;border-right:1px solid #e0dfd8;margin-right:8px;white-space:nowrap;flex-shrink:0}
.brand em{color:#e24b4a;font-style:normal}
.nav-btn{font-size:11px;font-weight:500;color:#888;padding:0 12px;height:48px;border:none;background:transparent;cursor:pointer;border-bottom:2px solid transparent;white-space:nowrap;margin-bottom:-1px;flex-shrink:0}
.nav-btn:hover{color:#333}
.nav-btn.on{color:#1a1a18;border-bottom-color:#e24b4a}
.sec{display:none;padding:18px 20px}
.sec.on{display:block}
.hero{background:#fff;border:1px solid #e0dfd8;border-radius:8px;padding:18px 22px;margin-bottom:14px}
.hero h1{font-size:17px;font-weight:600;margin-bottom:3px}
.hero h1 span{color:#e24b4a}
.hero p{font-size:11px;color:#888;margin-bottom:8px}
.tags{display:flex;gap:5px;flex-wrap:wrap}
.tag{font-size:10px;font-weight:500;padding:2px 8px;border-radius:4px}
.tag.r{background:#feecec;color:#a32d2d}
.tag.a{background:#fef3e0;color:#633806}
.tag.g{background:#eaf3de;color:#27500a}
.tag.p{background:#eeedfe;color:#3c3489}
.krow{display:grid;grid-template-columns:repeat(5,1fr);gap:9px;margin-bottom:14px}
.kpi{background:#fff;border:1px solid #e0dfd8;border-radius:8px;padding:12px 14px;border-top:3px solid #e0dfd8}
.kpi.r{border-top-color:#e24b4a}.kpi.a{border-top-color:#ef9f27}.kpi.g{border-top-color:#1d9e75}.kpi.p{border-top-color:#9b87f5}
.kl{font-size:9px;color:#888;text-transform:uppercase;letter-spacing:.04em;margin-bottom:5px}
.kv{font-size:20px;font-weight:600;line-height:1;margin-bottom:3px}
.kv.r{color:#a32d2d}.kv.a{color:#633806}.kv.g{color:#0f6e56}.kv.p{color:#3c3489}
.ks{font-size:10px;color:#aaa}
.g2{display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:12px}
.g3{display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;margin-bottom:12px}
.full{grid-column:1/-1}
.card{background:#fff;border:1px solid #e0dfd8;border-radius:8px;overflow:hidden}
.ch{padding:11px 14px 0;font-size:9px;font-weight:600;color:#888;text-transform:uppercase;letter-spacing:.05em}
.cw{position:relative;width:100%;padding:8px 14px 14px}
.leg{display:flex;flex-wrap:wrap;gap:9px;padding:4px 14px 0}
.li{display:flex;align-items:center;gap:4px;font-size:10px;color:#888}
.sq{width:7px;height:7px;border-radius:1px;flex-shrink:0}
.ins{background:#f0efe8;border-left:2px solid #9b87f5;padding:8px 12px;margin:8px 14px 14px;font-size:11px;color:#666;line-height:1.6;border-radius:0 4px 4px 0}
.ins strong{color:#1a1a18}
.hidden{display:none}
table{width:100%;border-collapse:collapse;font-size:11px}
th{font-size:9px;font-weight:600;color:#aaa;padding:6px 10px;border-bottom:1px solid #e0dfd8;text-align:left;text-transform:uppercase;letter-spacing:.04em}
td{padding:6px 10px;border-bottom:1px solid #f0efe8;color:#444}
tr:last-child td{border-bottom:none}
tr:hover td{background:#fafaf7}
.r{color:#a32d2d!important;font-weight:600}
.g{color:#0f6e56!important;font-weight:600}
.a{color:#633806!important;font-weight:600}
.nr{text-align:right;color:#a32d2d;font-weight:600}
.ng{text-align:right;color:#0f6e56;font-weight:600}
.na{text-align:right;color:#633806;font-weight:600}
.n{text-align:right}
.lbl{font-weight:500;color:#1a1a18}
.stabs{display:flex;border-bottom:1px solid #e0dfd8;padding:0 14px}
.stab{font-size:10px;font-weight:500;padding:7px 10px;cursor:pointer;border:none;background:transparent;color:#aaa;border-bottom:2px solid transparent;margin-bottom:-1px}
.stab:hover{color:#555}
.stab.on{color:#1a1a18;border-bottom-color:#9b87f5}
.sp{display:none}.sp.on{display:block}
.fl{display:flex;align-items:flex-start;gap:8px;padding:8px 14px;border-bottom:1px solid #f0efe8}
.fl:last-child{border-bottom:none}
.fp{font-size:9px;font-weight:700;padding:2px 5px;border-radius:3px;white-space:nowrap;flex-shrink:0;margin-top:1px;letter-spacing:.04em}
.fp.c{background:#feecec;color:#a32d2d}
.fp.al{background:#fef3e0;color:#633806}
.fp.at{background:#e6f1fb;color:#0c447c}
.fc{font-size:9px;padding:2px 5px;border-radius:3px;background:#f5f4f0;color:#888;white-space:nowrap;flex-shrink:0;margin-top:1px}
.ft{flex:1}
.ft-t{font-size:11px;font-weight:500;line-height:1.3;color:#1a1a18}
.ft-d{font-size:10px;color:#aaa;margin-top:2px}
.fv{font-size:11px;font-weight:600;white-space:nowrap}
.fv.r{color:#a32d2d}.fv.a{color:#633806}
.tl-item{display:flex;gap:10px;padding:8px 14px;border-bottom:1px solid #f0efe8;align-items:flex-start}
.tl-item:last-child{border-bottom:none}
.tl-dot{width:8px;height:8px;border-radius:50%;flex-shrink:0;margin-top:3px}
.tl-dot.p{background:#1d9e75}.tl-dot.m{background:#9b87f5}.tl-dot.al{background:#ef9f27}.tl-dot.r{background:#e24b4a}
.tl-date{font-size:10px;color:#888;min-width:62px;flex-shrink:0;margin-top:2px}
.tl-txt{font-size:11px;color:#555;line-height:1.45}
.cmp{display:grid;grid-template-columns:1fr 1fr 1fr}
.cmp-col{padding:14px;border-right:1px solid #e0dfd8}
.cmp-col:last-child{border-right:none}
.cmp-name{font-size:13px;font-weight:600;margin-bottom:2px}
.cmp-sub{font-size:9px;color:#aaa;text-transform:uppercase;letter-spacing:.04em;margin-bottom:8px}
.cmp-bdg{display:inline-block;font-size:9px;font-weight:600;padding:2px 6px;border-radius:3px;margin-bottom:7px}
.cmp-bdg.r{background:#feecec;color:#a32d2d}
.cmp-bdg.g{background:#eaf3de;color:#27500a}
.cmp-row{display:flex;justify-content:space-between;font-size:10px;padding:4px 0;border-bottom:1px solid #f5f4f0}
.cmp-row:last-child{border-bottom:none}
.cmp-rl{color:#888}.cmp-rv{font-weight:600}
.cmp-rv.r{color:#a32d2d}.cmp-rv.g{color:#0f6e56}.cmp-rv.a{color:#633806}
.sbar{display:flex;align-items:center;gap:8px;padding:4px 0;border-bottom:1px solid #f5f4f0}
.sbar:last-child{border-bottom:none}
.sbar-lbl{font-size:10px;color:#888;min-width:100px}
.sbar-bg{flex:1;height:5px;background:#f0efe8;border-radius:3px;overflow:hidden}
.sbar-fill{height:100%;border-radius:3px}
.sbar-val{font-size:10px;font-weight:600;min-width:34px;text-align:right}
.causa{display:flex;gap:10px;padding:11px 0;border-bottom:1px solid #f0efe8}
.causa:last-child{border-bottom:none}
.causa-n{font-size:18px;font-weight:700;color:#e24b4a;flex-shrink:0;width:22px;line-height:1}
.causa-t{font-size:12px;font-weight:600;margin-bottom:3px}
.causa-d{font-size:11px;color:#666;line-height:1.5}
@media(max-width:750px){.krow{grid-template-columns:1fr 1fr}.g2,.g3{grid-template-columns:1fr}.cmp{grid-template-columns:1fr}}
</style>
</head>
<body>
<nav class="nav">
  <div class="brand">RAIZ4 — <em>RE mar/2026</em></div>
  <button class="nav-btn on" onclick="nav('financeiro',this)">Financeiro</button>
  <button class="nav-btn" onclick="nav('estatistico',this)">Estatístico</button>
  <button class="nav-btn" onclick="nav('projecoes',this)">Projeções</button>
  <button class="nav-btn" onclick="nav('macro',this)">Macro</button>
  <button class="nav-btn" onclick="nav('concorrentes',this)">Concorrentes</button>
  <button class="nav-btn" onclick="nav('redflags',this)">Red Flags</button>
  <button class="nav-btn" onclick="nav('cronologia',this)">Cronologia</button>
  <button class="nav-btn" onclick="nav('resumo',this)">Resumo</button>
</nav>

<!-- ══ FINANCEIRO ══ -->
<div class="sec on" id="s-financeiro">
  <div class="hero">
    <h1>Raízen S.A. <span>RAIZ4</span> — Análise Financeira Completa</h1>
    <p>CNPJ 33.453.598/0001-23 · CVM/DFP consolidado · 2021/22→2024/25 · Ano fiscal abr–mar · Python puro · sem bibliotecas</p>
    <div class="tags">
      <span class="tag r">Recuperação extrajudicial mar/2026</span>
      <span class="tag r">R$ 98,6 bi — maior RE do Brasil</span>
      <span class="tag a">Selic 15% · Dívida R$ 70 bi</span>
      <span class="tag p">Altman Z 0,31 — colapso</span>
      <span class="tag g">Receita +20,5% mesmo assim</span>
    </div>
  </div>
  <div class="krow">
    <div class="kpi r"><div class="kl">Prejuízo 2024/25</div><div class="kv r">−R$ 17,2 bi</div><div class="ks">Margem −8,8%</div></div>
    <div class="kpi r"><div class="kl">Dívida bruta</div><div class="kv r">R$ 70 bi</div><div class="ks">+280% vs IPO</div></div>
    <div class="kpi a"><div class="kl">FCL acumulado 4 anos</div><div class="kv a">−R$ 24,2 bi</div><div class="ks">Negativo sempre</div></div>
    <div class="kpi r"><div class="kl">Juros pagos 2024/25</div><div class="kv r">R$ 12,1 bi</div><div class="ks">FCO só gera R$ 5,8 bi</div></div>
    <div class="kpi g"><div class="kl">Receita 2024/25</div><div class="kv g">R$ 194 bi</div><div class="ks">+20,5% vs 2021/22</div></div>
  </div>
  <div class="g2">
    <div class="card">
      <div class="ch">DRE — Resultado (R$ bi)</div>
      <div class="leg">
        <span class="li"><span class="sq" style="background:#1d9e75"></span>Receita</span>
        <span class="li"><span class="sq" style="background:#ef9f27"></span>EBITDA</span>
        <span class="li"><span class="sq" style="background:#e24b4a"></span>Lucro / Prejuízo</span>
      </div>
      <div class="cw" style="height:215px"><canvas id="c_dre"></canvas></div>
    </div>
    <div class="card">
      <div class="ch">DFC — Fluxo de Caixa (R$ bi)</div>
      <div class="leg">
        <span class="li"><span class="sq" style="background:#1d9e75"></span>FCO</span>
        <span class="li"><span class="sq" style="background:#ef9f27"></span>Capex</span>
        <span class="li"><span class="sq" style="background:#9b87f5"></span>Juros pagos</span>
        <span class="li"><span class="sq" style="background:#e24b4a"></span>FCL (linha)</span>
      </div>
      <div class="cw" style="height:215px"><canvas id="c_dfc"></canvas></div>
    </div>
  </div>
  <div class="g2">
    <div class="card">
      <div class="ch">Dívida bruta vs Patrimônio Líquido (R$ bi)</div>
      <div class="cw" style="height:200px"><canvas id="c_divida"></canvas></div>
    </div>
    <div class="card">
      <div class="ch">Indicadores — série histórica</div>
      <table id="tbl_ind"></table>
    </div>
  </div>
  <div class="ins"><strong>Paradoxo operacional:</strong> receita crescendo +20,5% com FCO positivo em todos os anos. A empresa não quebrou por falta de vendas. Quebrou por capex de R$ 52 bi em 4 anos + R$ 12,1 bi de juros em 2024/25 + impairment E2G de R$ 11,1 bi.</div>
</div>

<!-- ══ ESTATÍSTICO ══ -->
<div class="sec" id="s-estatistico">
  <div class="krow">
    <div class="kpi r"><div class="kl">Z-Score LL (2024/25)</div><div class="kv r" id="kv_zsll">—</div><div class="ks">Anomalia |Z|&gt;1.5</div></div>
    <div class="kpi r"><div class="kl">Z-Score dívida (2024/25)</div><div class="kv r" id="kv_zsdv">—</div><div class="ks">Outlier da própria série</div></div>
    <div class="kpi p"><div class="kl">Altman Z (2024/25)</div><div class="kv p" id="kv_az">—</div><div class="ks">Era 2,18 no IPO</div></div>
    <div class="kpi r"><div class="kl">Score de risco (2024/25)</div><div class="kv r" id="kv_sc">—</div><div class="ks">Cruzou 50 em 2022/23</div></div>
    <div class="kpi a"><div class="kl">Aviso metodológico</div><div class="kv a">n = 4</div><div class="ks">Indicadores direcionais</div></div>
  </div>
  <div class="g2">
    <div class="card">
      <div class="stabs">
        <button class="stab on" onclick="stab('zt','zs',this)">Z-Score estatístico</button>
        <button class="stab" onclick="stab('zt','altman',this)">Altman Z-Score</button>
      </div>
      <div id="zt-zs" class="sp on">
        <div class="leg" style="margin-top:8px">
          <span class="li"><span class="sq" style="background:#e24b4a"></span>Anomalia neg |Z|&gt;1.5</span>
          <span class="li"><span class="sq" style="background:#ef9f27"></span>Atenção</span>
          <span class="li"><span class="sq" style="background:#1d9e75"></span>Normal</span>
          <span class="li"><span class="sq" style="background:#378add"></span>Anomalia pos</span>
        </div>
        <div class="cw" style="height:200px"><canvas id="c_zs"></canvas></div>
        <table id="tbl_z" style="margin-bottom:8px"></table>
      </div>
      <div id="zt-altman" class="sp">
        <div class="leg" style="margin-top:8px">
          <span class="li"><span class="sq" style="background:#9b87f5"></span>Z calculado</span>
          <span class="li"><span class="sq" style="background:#1d9e75;opacity:.4"></span>Segura &gt;2,99</span>
          <span class="li"><span class="sq" style="background:#ef9f27;opacity:.4"></span>Cinza 1,81–2,99</span>
        </div>
        <div class="cw" style="height:200px"><canvas id="c_az"></canvas></div>
        <table id="tbl_az" style="margin-bottom:8px"></table>
      </div>
    </div>
    <div class="card">
      <div class="ch">Score de risco composto — 5 dimensões (0=saudável · 100=colapso)</div>
      <div class="cw" style="height:200px"><canvas id="c_sc1"></canvas></div>
      <div id="score-bars" style="padding:8px 14px 14px"></div>
    </div>
  </div>
  <div class="g2">
    <div class="card">
      <div class="ch">IQR — dívida bruta (R$ bi)</div>
      <div class="leg">
        <span class="li"><span class="sq" style="background:#e24b4a"></span>Outlier</span>
        <span class="li"><span class="sq" style="background:#378add"></span>Normal</span>
        <span class="li"><span class="sq" style="background:#ef9f27"></span>Limite IQR</span>
      </div>
      <div class="cw" style="height:195px"><canvas id="c_iq1"></canvas></div>
    </div>
    <div class="card">
      <div class="ch">IQR — cobertura de juros (×)</div>
      <div class="leg">
        <span class="li"><span class="sq" style="background:#e24b4a"></span>Abaixo do limite</span>
        <span class="li"><span class="sq" style="background:#1d9e75"></span>Normal</span>
      </div>
      <div class="cw" style="height:195px"><canvas id="c_iq2"></canvas></div>
    </div>
  </div>
</div>

<!-- ══ PROJEÇÕES ══ -->
<div class="sec" id="s-projecoes">
  <div class="krow">
    <div class="kpi r"><div class="kl">Dívida proj. 2025/26</div><div class="kv r" id="kv_pdv">—</div><div class="ks" id="kv_pdv_ic">IC 95%</div></div>
    <div class="kpi r"><div class="kl">LL proj. 2025/26</div><div class="kv r" id="kv_pll">—</div><div class="ks" id="kv_pll_ic">IC 95%</div></div>
    <div class="kpi r"><div class="kl">Cobertura proj. 2025/26</div><div class="kv r" id="kv_pcob">—</div><div class="ks">Insolvência de caixa</div></div>
    <div class="kpi g"><div class="kl">R² dívida (OLS)</div><div class="kv g" id="kv_r2dv">—</div><div class="ks">Crescimento estrutural</div></div>
    <div class="kpi a"><div class="kl">Selic sobre R$ 70 bi</div><div class="kv a">+R$ 700 mi</div><div class="ks">Por cada 1 p.p. adicional</div></div>
  </div>
  <div class="g2">
    <div class="card">
      <div class="ch">Projeção OLS — dívida e lucro com IC 95% (R$ bi)</div>
      <div class="leg">
        <span class="li"><span class="sq" style="background:#e24b4a"></span>Dívida real + proj.</span>
        <span class="li"><span class="sq" style="background:#378add"></span>LL real + proj.</span>
        <span class="li"><span class="sq" style="background:#ef9f27;opacity:.5"></span>IC ±1,96×RMSE</span>
      </div>
      <div class="cw" style="height:225px"><canvas id="c_proj1"></canvas></div>
    </div>
    <div class="card">
      <div class="ch">Cobertura de juros — 3 cenários (×)</div>
      <div class="leg">
        <span class="li"><span class="sq" style="background:#e24b4a"></span>Base (OLS)</span>
        <span class="li"><span class="sq" style="background:#1d9e75"></span>Recuperação</span>
        <span class="li"><span class="sq" style="background:#888"></span>Colapso</span>
      </div>
      <div class="cw" style="height:225px"><canvas id="c_cen"></canvas></div>
    </div>
  </div>
  <div class="g3">
    <div class="card">
      <div class="ch">Tabela de projeções OLS com IC 95%</div>
      <table id="tbl_proj" style="margin-bottom:8px"></table>
    </div>
    <div class="card">
      <div class="ch">Regressão OLS — todos os indicadores</div>
      <table id="tbl_ols" style="margin-bottom:8px"></table>
    </div>
    <div class="card">
      <div class="ch">Premissas dos cenários</div>
      <div class="fl"><div class="ft"><div class="ft-t">Base: Δdívida +25%/ano</div><div class="ft-d">Inclinação b da OLS da série de dívida · Selic 15%</div></div></div>
      <div class="fl"><div class="ft"><div class="ft-t">Recuperação: haircut 35%</div><div class="ft-d">Meta declarada pela Raízen na RE · FCO +10% · Selic 12%</div></div></div>
      <div class="fl"><div class="ft"><div class="ft-t">Colapso: Δdívida +40%/ano</div><div class="ft-d">Vencimentos antecipados · FCO −25% · Selic 15%</div></div></div>
      <div class="fl"><div class="ft"><div class="ft-t">Sensibilidade: +1 p.p. Selic</div><div class="ft-d">= +R$ 700 mi em juros/ano sobre R$ 70 bi</div></div></div>
    </div>
  </div>
  <div class="ins"><strong>Projeções via estatística própria:</strong> OLS implementado do zero em Python puro (sem numpy/scipy). IC 95% = ±1,96×RMSE calculado dos resíduos. Cenários derivados da inclinação b da própria regressão. Com n=4, use como cenário referência.</div>
</div>

<!-- ══ MACRO ══ -->
<div class="sec" id="s-macro">
  <div class="krow">
    <div class="kpi g"><div class="kl">CAGR milho (5 anos)</div><div class="kv g">+31%/ano</div><div class="ks">Cana: +0,5%/ano</div></div>
    <div class="kpi a"><div class="kl">Share milho 2024/25</div><div class="kv a">25%</div><div class="ks">Era 6% em 2019/20</div></div>
    <div class="kpi a"><div class="kl">Projeção milho 2030</div><div class="kv a">40%</div><div class="ks">BTG Pactual + Datagro</div></div>
    <div class="kpi r"><div class="kl">Gap custo 2024/25</div><div class="kv r">R$ 0,48/L</div><div class="ks">Era R$ 0,05/L em 2019/20</div></div>
    <div class="kpi r"><div class="kl">TIR E2G Raízen</div><div class="kv r">Negativa</div><div class="ks">TIR milho: 15,5% (BTG)</div></div>
  </div>
  <div class="g2">
    <div class="card">
      <div class="ch">Produção de etanol — cana vs milho (bi litros)</div>
      <div class="leg">
        <span class="li"><span class="sq" style="background:#1d9e75"></span>Cana</span>
        <span class="li"><span class="sq" style="background:#ef9f27"></span>Milho</span>
        <span class="li"><span class="sq" style="background:#9b87f5"></span>Share milho % (eixo dir.)</span>
      </div>
      <div class="cw" style="height:220px"><canvas id="c_macro"></canvas></div>
    </div>
    <div class="card">
      <div class="ch">Custo de produção — milho vs cana (R$/L)</div>
      <div class="leg">
        <span class="li"><span class="sq" style="background:#ef9f27"></span>Milho</span>
        <span class="li"><span class="sq" style="background:#e24b4a"></span>Cana</span>
      </div>
      <div class="cw" style="height:220px"><canvas id="c_custos"></canvas></div>
    </div>
  </div>
  <div class="g3">
    <div class="card">
      <div class="ch">Projeções de market share</div>
      <table>
        <thead><tr><th>Safra</th><th>Milho bi L</th><th>Share</th></tr></thead>
        <tbody>
          <tr><td>2019/20</td><td class="n">2,0</td><td class="n">6,3%</td></tr>
          <tr><td>2024/25</td><td class="na">10,2</td><td class="na">25,1%</td></tr>
          <tr><td>2030p BTG</td><td class="ng">18,0</td><td class="ng">40,0%</td></tr>
          <tr><td>2034p Datagro</td><td class="ng">24,7</td><td class="ng">49,0%</td></tr>
        </tbody>
      </table>
    </div>
    <div class="card">
      <div class="ch">Vantagens estruturais do milho</div>
      <table>
        <thead><tr><th>Dimensão</th><th>Milho</th><th>Cana/E2G</th></tr></thead>
        <tbody>
          <tr><td class="lbl">Sazonalidade</td><td class="g">12 meses</td><td class="r">6–7 meses</td></tr>
          <tr><td class="lbl">Armazenagem</td><td class="g">Meses</td><td class="r">48h</td></tr>
          <tr><td class="lbl">Hedge natural</td><td class="g">DDG + óleo</td><td class="r">Nenhum</td></tr>
          <tr><td class="lbl">Capex/litro</td><td class="g">R$ 1,5–2,0</td><td class="r">R$ 5–8</td></tr>
          <tr><td class="lbl">TIR estimada</td><td class="g">15,5%</td><td class="r">Negativa</td></tr>
        </tbody>
      </table>
    </div>
    <div class="card">
      <div class="ch">CAGR comparado (5 anos)</div>
      <div id="cagr-bars" style="padding:12px 14px"></div>
    </div>
  </div>
  <div class="ins"><strong>O timing foi fatal:</strong> R$ 36+ bi comprometidos em E2G entre 2021–2024 — exatamente quando o milho crescia 31%/ano. Em 2023 o mercado se recusou a pagar prêmio ambiental pelo E2G — a premissa fundamental foi refutada.</div>
</div>

<!-- ══ CONCORRENTES ══ -->
<div class="sec" id="s-concorrentes">
  <div class="krow">
    <div class="kpi r"><div class="kl">Margem liq. Raízen</div><div class="kv r">−8,8%</div><div class="ks">RE mar/2026</div></div>
    <div class="kpi g"><div class="kl">Margem liq. Inpasa</div><div class="kv g">+18,4%</div><div class="ks">CAGR receita +50%/ano</div></div>
    <div class="kpi g"><div class="kl">Margem liq. FS Bio</div><div class="kv g">+8,8%</div><div class="ks">3ª maior do Brasil</div></div>
    <div class="kpi r"><div class="kl">Diferença de margem</div><div class="kv r">27,2 p.p.</div><div class="ks">Raízen vs Inpasa</div></div>
    <div class="kpi r"><div class="kl">Alavancagem Raízen</div><div class="kv r">5,3×</div><div class="ks">vs Inpasa: 1,7×</div></div>
  </div>
  <div class="card" style="margin-bottom:12px">
    <div class="ch">Comparativo — mesma commodity, resultados opostos</div>
    <div class="cmp">
      <div class="cmp-col" style="background:#feecec08">
        <div class="cmp-bdg r">Recuperação extrajudicial</div>
        <div class="cmp-name">Raízen S.A.</div>
        <div class="cmp-sub">Cana + E2G · RAIZ4 · Listada</div>
        <div class="cmp-row"><span class="cmp-rl">Receita</span><span class="cmp-rv a">R$ 194 bi</span></div>
        <div class="cmp-row"><span class="cmp-rl">Lucro líquido</span><span class="cmp-rv r">−R$ 17,2 bi</span></div>
        <div class="cmp-row"><span class="cmp-rl">Margem líquida</span><span class="cmp-rv r">−8,8%</span></div>
        <div class="cmp-row"><span class="cmp-rl">Alavancagem</span><span class="cmp-rv r">5,3×</span></div>
        <div class="cmp-row"><span class="cmp-rl">Custo/litro</span><span class="cmp-rv r">R$ 2,36</span></div>
        <div class="cmp-row"><span class="cmp-rl">TIR E2G</span><span class="cmp-rv r">Negativa</span></div>
        <div class="cmp-row"><span class="cmp-rl">Capex 4 anos</span><span class="cmp-rv r">R$ 52 bi</span></div>
      </div>
      <div class="cmp-col">
        <div class="cmp-bdg g">Líder América Latina</div>
        <div class="cmp-name">Inpasa Agroindustrial</div>
        <div class="cmp-sub">Etanol de milho · Privada</div>
        <div class="cmp-row"><span class="cmp-rl">Receita</span><span class="cmp-rv g">R$ 13,6 bi</span></div>
        <div class="cmp-row"><span class="cmp-rl">Lucro líquido</span><span class="cmp-rv g">+R$ 2,5 bi</span></div>
        <div class="cmp-row"><span class="cmp-rl">Margem líquida</span><span class="cmp-rv g">+18,4%</span></div>
        <div class="cmp-row"><span class="cmp-rl">Alavancagem</span><span class="cmp-rv g">1,7×</span></div>
        <div class="cmp-row"><span class="cmp-rl">Custo/litro</span><span class="cmp-rv g">R$ 1,88</span></div>
        <div class="cmp-row"><span class="cmp-rl">TIR</span><span class="cmp-rv g">15,5%</span></div>
        <div class="cmp-row"><span class="cmp-rl">CAGR receita 5a</span><span class="cmp-rv g">+50%/ano</span></div>
      </div>
      <div class="cmp-col">
        <div class="cmp-bdg g">Em expansão</div>
        <div class="cmp-name">FS Bioenergia</div>
        <div class="cmp-sub">Etanol de milho · Privada</div>
        <div class="cmp-row"><span class="cmp-rl">Receita</span><span class="cmp-rv g">R$ 10,7 bi</span></div>
        <div class="cmp-row"><span class="cmp-rl">Lucro líquido</span><span class="cmp-rv g">+R$ 0,94 bi</span></div>
        <div class="cmp-row"><span class="cmp-rl">Margem líquida</span><span class="cmp-rv g">+8,8%</span></div>
        <div class="cmp-row"><span class="cmp-rl">Alavancagem</span><span class="cmp-rv">N/D</span></div>
        <div class="cmp-row"><span class="cmp-rl">Custo/litro</span><span class="cmp-rv g">R$ 1,88</span></div>
        <div class="cmp-row"><span class="cmp-rl">TIR</span><span class="cmp-rv g">~15,5%</span></div>
        <div class="cmp-row"><span class="cmp-rl">Produção</span><span class="cmp-rv">2,4 bi L</span></div>
      </div>
    </div>
  </div>
  <div class="g2">
    <div class="card">
      <div class="ch">Margem líquida e alavancagem — comparativo</div>
      <div class="cw" style="height:200px"><canvas id="c_cmp"></canvas></div>
    </div>
    <div class="card">
      <div class="ch">Inpasa — crescimento de receita (R$ bi)</div>
      <div class="cw" style="height:200px"><canvas id="c_inp"></canvas></div>
    </div>
  </div>
  <div class="ins"><strong>A diferença não é de setor — é de modelo de negócio.</strong> Inpasa: tecnologia simples + custo baixo + expansão sustentável. Raízen: tecnologia complexa + aposta em prêmio ambiental que o mercado se recusou a pagar. Inpasa cresceu de R$ 376 mi para R$ 13,6 bi em 5 anos (CAGR 50%/ano).</div>
</div>

<!-- ══ RED FLAGS ══ -->
<div class="sec" id="s-redflags">
  <div class="krow">
    <div class="kpi r"><div class="kl">Total de red flags</div><div class="kv r">38</div><div class="ks">28 financeiros + 10 setoriais</div></div>
    <div class="kpi r"><div class="kl">Flags críticos</div><div class="kv r">24</div><div class="ks">DRE + Balanço + DFC + Setorial</div></div>
    <div class="kpi a"><div class="kl">1º sinal setorial</div><div class="kv a">2019/20</div><div class="ks">Milho crescendo 100%/ano</div></div>
    <div class="kpi a"><div class="kl">1º flag financeiro</div><div class="kv a">2021/22</div><div class="ks">FCL negativo no ano do IPO</div></div>
    <div class="kpi r"><div class="kl">Resultado final</div><div class="kv r">Mar/2026</div><div class="ks">Maior RE da história BR</div></div>
  </div>
  <div class="g2">
    <div class="card">
      <div class="ch">Red flags financeiros — detecção automática Python</div>
      <div id="flags-fin"></div>
    </div>
    <div class="card">
      <div class="ch">Red flags setoriais — sinais do mercado</div>
      <div class="fl"><span class="fp at">ATENÇÃO</span><span class="fc">MACRO</span><div class="ft"><div class="ft-t">Milho sai de 2% para 8% do mercado em 2 anos (2019–21)</div><div class="ft-d">CAGR ~100%/ano — disrupção nascente visível nos dados</div></div></div>
      <div class="fl"><span class="fp al">ALERTA</span><span class="fc">ESTRATÉGICO</span><div class="ft"><div class="ft-t">IPO capta R$ 6,9 bi e anuncia expansão E2G (ago/2021)</div><div class="ft-d">Sem prova de prêmio ambiental no mercado spot</div></div></div>
      <div class="fl"><span class="fp al">ALERTA</span><span class="fc">CUSTO</span><div class="ft"><div class="ft-t">Gap de custo cresce de R$ 0,05 para R$ 0,12/L (2022/23)</div><div class="ft-d">Vantagem estrutural do milho consolidando</div></div></div>
      <div class="fl"><span class="fp c">CRÍTICO</span><span class="fc">ESTRATÉGICO</span><div class="ft"><div class="ft-t">Mercado não paga prêmio ambiental pelo E2G (2023)</div><div class="ft-d">Premissa de R$ 36 bi refutada pelo próprio mercado</div></div></div>
      <div class="fl"><span class="fp c">CRÍTICO</span><span class="fc">OPERACIONAL</span><div class="ft"><div class="ft-t">Incêndios destroem ~30% das lavouras de cana (2023)</div><div class="ft-d">E2G depende do bagaço — sem hedge de matéria-prima</div></div></div>
      <div class="fl"><span class="fp c">CRÍTICO</span><span class="fc">OPERACIONAL</span><div class="ft"><div class="ft-t">Impairment E2G: R$ 11,1 bi (3T 2025/26)</div><div class="ft-d">Ativos valem menos que o custo de construção</div></div></div>
      <div class="fl"><span class="fp c">CRÍTICO</span><span class="fc">MACRO</span><div class="ft"><div class="ft-t">Selic 15% · R$ 70 bi × 15% = R$ 10,5 bi/ano (2024/25)</div><div class="ft-d">+1 p.p. = +R$ 700 mi em juros sobre a dívida atual</div></div></div>
      <div class="fl"><span class="fp c">CRÍTICO</span><span class="fc">CUSTO</span><div class="ft"><div class="ft-t">Gap explode para R$ 0,48/L — milho 20% mais barato</div><div class="ft-d">TIR milho 15,5% vs E2G negativa: inversão total</div></div></div>
      <div class="fl"><span class="fp c">CRÍTICO</span><span class="fc">RESULTADO</span><div class="ft"><div class="ft-t">Maior RE da história do Brasil (mar/2026)</div><div class="ft-d">R$ 98,6 bi total · R$ 65,1 bi financeira quirografária</div></div></div>
    </div>
  </div>
</div>

<!-- ══ CRONOLOGIA ══ -->
<div class="sec" id="s-cronologia">
  <div class="krow">
    <div class="kpi p"><div class="kl">IPO → Altman zona perigo</div><div class="kv p">18 meses</div><div class="ks">ago/2021 → 2022/23</div></div>
    <div class="kpi a"><div class="kl">IPO → 1º prejuízo</div><div class="kv a">30 meses</div><div class="ks">ago/2021 → 2023/24</div></div>
    <div class="kpi r"><div class="kl">Score &gt;50 → RE</div><div class="kv r">24 meses</div><div class="ks">2022/23 → mar/2026</div></div>
    <div class="kpi r"><div class="kl">IPO → RE</div><div class="kv r">55 meses</div><div class="ks">ago/2021 → mar/2026</div></div>
    <div class="kpi r"><div class="kl">Rating perdido</div><div class="kv r">9 níveis</div><div class="ks">Moody's Baa3→Ca em 4 anos</div></div>
  </div>
  <div class="g2">
    <div class="card">
      <div class="ch">Linha do tempo — do IPO à recuperação extrajudicial</div>
      <div>
        <div class="tl-item"><span class="tl-dot p"></span><span class="tl-date">2011</span><span class="tl-txt">Criação da Raízen — JV Cosan + Shell. Rating Baa3 (grau de investimento)</span></div>
        <div class="tl-item"><span class="tl-dot p"></span><span class="tl-date">2015</span><span class="tl-txt">Inaugura 1ª planta E2G do mundo. Aposta tecnológica validada internamente</span></div>
        <div class="tl-item"><span class="tl-dot m"></span><span class="tl-date">Ago/2021</span><span class="tl-txt">IPO na B3 — maior do setor de energia. Capta R$ 6,9 bi. Anuncia expansão massiva em E2G</span></div>
        <div class="tl-item"><span class="tl-dot al"></span><span class="tl-date">2021/22</span><span class="tl-txt">FCL: −R$ 3,6 bi no 1º ano listado. Altman Z: 2,18 (zona cinza). Score: 32/100</span></div>
        <div class="tl-item"><span class="tl-dot al"></span><span class="tl-date">2022</span><span class="tl-txt">Selic sobe para 13,75%. Despesa financeira: R$ 5,2 bi. Dívida dobra para R$ 37 bi</span></div>
        <div class="tl-item"><span class="tl-dot al"></span><span class="tl-date">2022/23</span><span class="tl-txt">Altman Z: 1,52 — entra zona de perigo. Score cruza 50/100. Cobertura: 1,48×</span></div>
        <div class="tl-item"><span class="tl-dot r"></span><span class="tl-date">2023</span><span class="tl-txt">Incêndios destroem ~30% das lavouras de cana. Milho já tem 15% do mercado</span></div>
        <div class="tl-item"><span class="tl-dot r"></span><span class="tl-date">2023</span><span class="tl-txt">Mercado recusa pagar prêmio ambiental pelo E2G — premissa de R$ 36 bi cai</span></div>
        <div class="tl-item"><span class="tl-dot r"></span><span class="tl-date">2023/24</span><span class="tl-txt">1º prejuízo: −R$ 3,1 bi. Liq. corrente &lt;1,0×. Dívida: R$ 52 bi. Score: 69/100</span></div>
        <div class="tl-item"><span class="tl-dot r"></span><span class="tl-date">Nov/2024</span><span class="tl-txt">Vende usinas. Troca CEO e CFO. Mercado precifica probabilidade de RE</span></div>
        <div class="tl-item"><span class="tl-dot r"></span><span class="tl-date">Nov/2025</span><span class="tl-txt">Moody's retira grau de investimento. Rating cai 9 níveis em 4 anos</span></div>
        <div class="tl-item"><span class="tl-dot r"></span><span class="tl-date">Fev/2026</span><span class="tl-txt">Prejuízo R$ 15,6 bi no 3T. Impairment E2G: R$ 11,1 bi. PL negativo. Score: 86/100</span></div>
        <div class="tl-item"><span class="tl-dot r"></span><span class="tl-date">Mar/2026</span><span class="tl-txt">RE — maior da história do Brasil. R$ 98,6 bi total · R$ 65,1 bi financeira</span></div>
        <div class="tl-item"><span class="tl-dot r"></span><span class="tl-date">Mar/2026</span><span class="tl-txt">Shell aporta R$ 3,5 bi. Proposta: conversão 40% da dívida em equity</span></div>
      </div>
    </div>
    <div class="card">
      <div class="ch">Score de risco + Altman Z — convergência</div>
      <div class="leg">
        <span class="li"><span class="sq" style="background:#e24b4a"></span>Score /100 (eixo esq.)</span>
        <span class="li"><span class="sq" style="background:#9b87f5"></span>Altman Z (eixo dir.)</span>
      </div>
      <div class="cw" style="height:200px"><canvas id="c_conv"></canvas></div>
      <table style="margin-bottom:8px">
        <thead><tr><th>Marco</th><th>Antecipação à RE</th></tr></thead>
        <tbody>
          <tr><td class="lbl">1º sinal setorial (milho)</td><td class="na">~5 anos</td></tr>
          <tr><td class="lbl">Altman Z zona perigo</td><td class="na">3 anos</td></tr>
          <tr><td class="lbl">Score &gt;50/100</td><td class="nr">2 anos</td></tr>
          <tr><td class="lbl">Cobertura juros &lt;1×</td><td class="nr">2 anos</td></tr>
        </tbody>
      </table>
    </div>
  </div>
</div>

<!-- ══ RESUMO ══ -->
<div class="sec" id="s-resumo">
  <div class="krow">
    <div class="kpi r"><div class="kl">Red flags totais</div><div class="kv r">38</div><div class="ks">Financeiros + setoriais</div></div>
    <div class="kpi r"><div class="kl">Variação lucro líquido</div><div class="kv r">−919%</div><div class="ks">+R$ 2,1 bi → −R$ 17,2 bi</div></div>
    <div class="kpi r"><div class="kl">Variação dívida</div><div class="kv r">+280%</div><div class="ks">R$ 18,4 bi → R$ 70 bi</div></div>
    <div class="kpi a"><div class="kl">1º sinal detectável</div><div class="kv a">2019/20</div><div class="ks">Milho crescendo 100%/ano</div></div>
    <div class="kpi p"><div class="kl">Score &gt;50 → RE</div><div class="kv p">24 meses</div><div class="ks">Modelos sinalizaram antes</div></div>
  </div>
  <div class="g2">
    <div class="card">
      <div class="ch">As 3 causas raiz — análise integrada</div>
      <div style="padding:14px">
        <div class="causa"><div class="causa-n">1</div><div><div class="causa-t">Aposta estratégica errada no E2G</div><div class="causa-d">R$ 36+ bi em tecnologia sem demanda de mercado confirmada. TIR negativa. O mercado nunca pagou o prêmio ambiental. Impairment de R$ 11,1 bi reconhecido em fev/2026.</div></div></div>
        <div class="causa"><div class="causa-n">2</div><div><div class="causa-t">Estrutura de capital frágil desde o IPO</div><div class="causa-d">Alavancagem de 2,1× no 1º ano listado, crescendo para 5,3× em 4 anos. Selic 15% sobre R$ 70 bi = R$ 10,5 bi/ano só em juros — destruindo todo o FCO de R$ 5,8 bi.</div></div></div>
        <div class="causa"><div class="causa-n">3</div><div><div class="causa-t">Disrupção externa do milho subestimada</div><div class="causa-d">Milho cresceu 31%/ano por 5 anos. Gap de custo: R$ 0,05 → R$ 0,48/litro (+860%). Todo o capital novo do setor foi para o milho — E2G ficou sem financiamento.</div></div></div>
      </div>
    </div>
    <div class="card">
      <div class="ch">Sinais detectáveis antecipadamente</div>
      <table>
        <thead><tr><th>Sinal</th><th>Quando</th><th>Fonte</th><th>Tipo</th></tr></thead>
        <tbody>
          <tr><td class="lbl">Milho CAGR 100%/ano</td><td class="n">2019–21</td><td>UNEM/Datagro</td><td class="a">Setorial</td></tr>
          <tr><td class="lbl">FCL negativo desde IPO</td><td class="n">2021/22</td><td>DFC/CVM</td><td class="a">DFC</td></tr>
          <tr><td class="lbl">Altman Z zona cinza</td><td class="n">2021/22</td><td>Cálc. próprio</td><td class="a">Estat.</td></tr>
          <tr><td class="lbl">Score cruza 50/100</td><td class="n">2022/23</td><td>Cálc. próprio</td><td class="r">Score</td></tr>
          <tr><td class="lbl">Altman Z zona perigo</td><td class="n">2022/23</td><td>Cálc. próprio</td><td class="r">Estat.</td></tr>
          <tr><td class="lbl">Cobertura juros &lt;1×</td><td class="n">2023/24</td><td>DFC/CVM</td><td class="r">DFC</td></tr>
          <tr><td class="lbl">PL negativo</td><td class="n">2024/25</td><td>BP/CVM</td><td class="r">Balanço</td></tr>
          <tr><td class="lbl">RE</td><td class="n">Mar/2026</td><td>Junta Comercial</td><td class="r">Resultado</td></tr>
        </tbody>
      </table>
    </div>
  </div>
  <div class="g3">
    <div class="card">
      <div class="ch">Estatísticas finais</div>
      <table>
        <thead><tr><th>Métrica</th><th>Valor</th></tr></thead>
        <tbody id="tbl_est"></tbody>
      </table>
    </div>
    <div class="card">
      <div class="ch">Modelos próprios — antecipação</div>
      <table>
        <thead><tr><th>Modelo</th><th>Sinal</th><th>Antecipação</th></tr></thead>
        <tbody>
          <tr><td class="lbl">Altman Z-Score</td><td class="a">Zona perigo</td><td class="na">3 anos antes</td></tr>
          <tr><td class="lbl">Score composto</td><td class="a">Cruza 50/100</td><td class="na">2 anos antes</td></tr>
          <tr><td class="lbl">IQR dívida</td><td class="r">Outlier alto</td><td class="nr">2024/25</td></tr>
          <tr><td class="lbl">Z-Score LL</td><td class="r">Anomalia −1,74σ</td><td class="nr">2024/25</td></tr>
          <tr><td class="lbl">OLS cobertura</td><td class="r">Proj. 0,32×</td><td class="nr">2025/26p</td></tr>
        </tbody>
      </table>
    </div>
    <div class="card">
      <div class="ch">Conclusão</div>
      <div style="padding:14px;font-size:12px;color:#444;line-height:1.7">
        <p style="margin-bottom:10px">A Raízen não quebrou por incompetência operacional — <strong>FCO positivo em todos os 4 anos</strong>. Quebrou pela confluência de três forças simultâneas.</p>
        <p><strong>Todos os três sinais eram detectáveis antes de 2023</strong> — pelo Altman Z-Score, pelo score de risco próprio e pelo CAGR do mercado de milho. Os dados estavam disponíveis. O mercado de E2G não estava.</p>
      </div>
    </div>
  </div>
</div>

<script>
// ── DADOS PRÉ-CALCULADOS EM PYTHON ────────────────────────────────────────────
const PY = __DADOS_JSON__;

// ── ATALHOS ───────────────────────────────────────────────────────────────────
const AN = PY.AN;
const ANp = PY.ANp;

// ── PREENCHER KPIs COM DADOS CALCULADOS ──────────────────────────────────────
document.getElementById('kv_zsll').textContent = PY.zs_ll[3].toFixed(2) + 'σ';
document.getElementById('kv_zsdv').textContent = '+' + PY.zs_dv[3].toFixed(2) + 'σ';
document.getElementById('kv_az').textContent   = PY.altmans[3].Z.toFixed(2);
document.getElementById('kv_sc').textContent   = PY.scores[3].comp.toFixed(1) + '/100';
document.getElementById('kv_pdv').textContent  = '~R$ ' + PY.proj_dv[0].yp.toFixed(1) + ' bi';
document.getElementById('kv_pdv_ic').textContent = 'IC 95%: R$ ' + PY.proj_dv[0].lo.toFixed(0) + '–' + PY.proj_dv[0].hi.toFixed(0) + ' bi';
document.getElementById('kv_pll').textContent  = '~−R$ ' + Math.abs(PY.proj_ll[0].yp).toFixed(1) + ' bi';
document.getElementById('kv_pll_ic').textContent = 'IC 95%: ' + PY.proj_ll[0].lo.toFixed(0) + ' a ' + PY.proj_ll[0].hi.toFixed(0) + ' bi';
document.getElementById('kv_pcob').textContent = PY.proj_cob[0].yp.toFixed(2) + '×';
document.getElementById('kv_r2dv').textContent = PY.ols_dv.r2.toFixed(4);

// ── BASE CHART.JS ─────────────────────────────────────────────────────────────
const B = {
  responsive: true, maintainAspectRatio: false,
  plugins: { legend: { display: false } },
  scales: {
    x: { grid: { display: false }, ticks: { color: '#aaa', font: { size: 10 } } },
    y: { grid: { color: 'rgba(0,0,0,.05)' }, ticks: { color: '#aaa', font: { size: 10 } } }
  }
};
function mk(id, type, labels, datasets, ex) {
  const el = document.getElementById(id);
  if (!el) return;
  try { new Chart(el, { type, data: { labels, datasets }, options: Object.assign({}, B, ex) }); }
  catch(e) { console.warn(id, e); }
}

// ── FINANCEIRO ────────────────────────────────────────────────────────────────
mk('c_dre', 'line', AN, [
  { data: PY.receita, borderColor:'#1d9e75', borderWidth:2, fill:false, tension:.4, pointRadius:4, pointBackgroundColor:'#1d9e75' },
  { data: PY.ebitda,  borderColor:'#ef9f27', borderWidth:2, fill:false, tension:.4, pointRadius:4, pointBackgroundColor:'#ef9f27' },
  { data: PY.lucro_liq, borderColor:'#e24b4a', borderWidth:2.5, fill:false, tension:.4, pointRadius:5,
    pointBackgroundColor: PY.lucro_liq.map(v => v >= 0 ? '#e24b4a' : '#8b0000') },
], { scales:{ x:{grid:{display:false},ticks:{color:'#aaa',font:{size:10}}}, y:{grid:{color:'rgba(0,0,0,.05)'},ticks:{color:'#aaa',font:{size:10},callback:v=>v+'bi'}} } });

mk('c_dfc', 'bar', AN, [
  { data: PY.fco,                  backgroundColor:'rgba(29,158,117,.7)', borderRadius:3, barPercentage:.25, stack:'a' },
  { data: PY.capex.map(v => -v),   backgroundColor:'rgba(239,159,39,.7)', borderRadius:3, barPercentage:.25, stack:'a' },
  { data: PY.juros_pagos.map(v=>-v), backgroundColor:'rgba(155,135,245,.7)', borderRadius:3, barPercentage:.25, stack:'a' },
  { data: PY.fcl, type:'line', borderColor:'#e24b4a', borderWidth:2.5, fill:false, tension:.4, pointRadius:5, pointBackgroundColor:'#e24b4a' },
], { scales:{ x:{stacked:true,grid:{display:false},ticks:{color:'#aaa',font:{size:10}}}, y:{stacked:true,grid:{color:'rgba(0,0,0,.05)'},ticks:{color:'#aaa',font:{size:10},callback:v=>v+'bi'}} }, plugins:{legend:{display:false}} });

mk('c_divida', 'bar', AN, [
  { data: PY.divida_bruta, backgroundColor:'rgba(226,75,74,.7)', borderRadius:3, barPercentage:.35 },
  { data: PY.pl.map(v => Math.max(v,0)), backgroundColor:'rgba(155,135,245,.7)', borderRadius:3, barPercentage:.35 },
  { data: PY.pl.map(v => Math.min(v,0)), backgroundColor:'rgba(226,75,74,.35)', borderRadius:3, barPercentage:.35 },
], { scales:{ x:{grid:{display:false},ticks:{color:'#aaa',font:{size:10}}}, y:{grid:{color:'rgba(0,0,0,.05)'},ticks:{color:'#aaa',font:{size:10},callback:v=>v+'bi'}} } });

// Tabela de indicadores financeiros
(function() {
  const el = document.getElementById('tbl_ind');
  if (!el) return;
  const th = document.createElement('thead');
  th.innerHTML = '<tr><th>Indicador</th>' + AN.map(a=>`<th style="text-align:right">${a}</th>`).join('') + '</tr>';
  el.appendChild(th);
  const tb = document.createElement('tbody');
  const rows = [
    ['Margem bruta %',   PY.mg_bruta,  'a'],
    ['Margem EBIT %',    PY.mg_ebit,   'r'],
    ['Margem líq. %',    PY.mg_liq,    'r'],
    ['Liq. corrente ×',  PY.liq_cor,   'r'],
    ['Alavancagem ×',    PY.alav,      'r'],
    ['Endividamento %',  PY.end_ger,   'r'],
    ['Cobertura juros ×',PY.cobert_j,  'r'],
    ['ROA %',            PY.roa,       'r'],
    ['FCO (bi)',          PY.fco,       'n'],
    ['FCL (bi)',          PY.fcl,       'r'],
  ];
  rows.forEach(([n, d, cls]) => {
    const tr = document.createElement('tr');
    tr.innerHTML = `<td class="lbl">${n}</td>` + d.map((v,i) => {
      const negative = (cls === 'r' && (v < 0 || (n.includes('Marg') && v < 0) || (n.includes('cob') && v < 1) || (n.includes('Liq') && v < 1)));
      const c = v < 0 ? 'nr' : (cls === 'a' ? 'na' : 'n');
      return `<td class="${c}">${v.toFixed(2)}</td>`;
    }).join('');
    tb.appendChild(tr);
  });
  el.appendChild(tb);
})();

// ── ESTATÍSTICO ───────────────────────────────────────────────────────────────
mk('c_zs', 'bar', AN, [
  { data: PY.zs_ll,  backgroundColor: PY.zs_ll.map(z=>Math.abs(z)>1.5?'#e24b4a':Math.abs(z)>1?'#ef9f27':'#1d9e75'), borderRadius:4, barPercentage:.3 },
  { data: PY.zs_dv,  backgroundColor: PY.zs_dv.map(z=>z>1.5?'#378add':Math.abs(z)>1?'#ef9f27':'#1d9e75'), borderRadius:4, barPercentage:.3 },
  { data: PY.zs_cob, backgroundColor: PY.zs_cob.map(z=>Math.abs(z)>1.5?'#e24b4a':Math.abs(z)>1?'#ef9f27':'#1d9e75'), borderRadius:4, barPercentage:.3 },
], { scales:{ x:{grid:{display:false},ticks:{color:'#aaa',font:{size:10}}}, y:{grid:{color:'rgba(0,0,0,.05)'},ticks:{color:'#aaa',font:{size:10},callback:v=>v.toFixed(1)+'σ'}} } });

// Tabela Z-Score
(function() {
  const el = document.getElementById('tbl_z');
  if (!el) return;
  el.innerHTML = '<thead><tr><th>Indicador</th>' + AN.map(a=>`<th style="text-align:right">${a}</th>`).join('') + '</tr></thead>';
  const tb = document.createElement('tbody');
  [['Lucro líq.', PY.zs_ll], ['Dívida', PY.zs_dv], ['Juros', PY.zs_jr],
   ['Cobertura', PY.zs_cob], ['FCO', PY.zs_fco], ['Alavancagem', PY.zs_alav]].forEach(([n,z]) => {
    const tr = document.createElement('tr');
    tr.innerHTML = `<td class="lbl">${n}</td>` + z.map(v => {
      const cls = Math.abs(v) > 1.5 ? (v < 0 ? 'nr' : 'ng') : Math.abs(v) > 1 ? 'na' : 'n';
      return `<td class="${cls}">${v > 0 ? '+' : ''}${v.toFixed(2)}σ</td>`;
    }).join('');
    tb.appendChild(tr);
  });
  el.appendChild(tb);
})();

mk('c_az', 'line', AN, [
  { data: PY.altmans.map(r=>r.Z), borderColor:'#9b87f5', borderWidth:2.5, fill:false, tension:.4,
    pointRadius:6, pointBackgroundColor: PY.altmans.map(r=>r.Z>2.99?'#1d9e75':r.Z>=1.81?'#ef9f27':'#e24b4a') },
  { data:[2.99,2.99,2.99,2.99], borderColor:'rgba(29,158,117,.4)', borderWidth:1, borderDash:[4,3], pointRadius:0, fill:false },
  { data:[1.81,1.81,1.81,1.81], borderColor:'rgba(239,159,39,.4)', borderWidth:1, borderDash:[4,3], pointRadius:0, fill:false },
], { scales:{ x:{grid:{display:false},ticks:{color:'#aaa',font:{size:10}}}, y:{grid:{color:'rgba(0,0,0,.05)'},ticks:{color:'#aaa',font:{size:10},callback:v=>v.toFixed(2)},min:-0.3,max:3.4} } });

(function() {
  const el = document.getElementById('tbl_az');
  if (!el) return;
  el.innerHTML = '<thead><tr><th>Var.</th>' + AN.map(a=>`<th style="text-align:right">${a}</th>`).join('') + '</tr></thead>';
  const tb = document.createElement('tbody');
  [['X1',r=>r.X1],['X2',r=>r.X2],['X3',r=>r.X3],['X4',r=>r.X4],['X5',r=>r.X5],['Z',r=>r.Z]].forEach(([n,fn]) => {
    const tr = document.createElement('tr');
    tr.innerHTML = `<td class="lbl">${n}</td>` + PY.altmans.map(r => {
      const v = fn(r); const cls = v >= 0 ? 'ng' : 'nr';
      return `<td class="${cls}">${v > 0 ? '+' : ''}${v.toFixed(3)}</td>`;
    }).join('');
    tb.appendChild(tr);
  });
  const trZ = document.createElement('tr');
  trZ.innerHTML = '<td class="lbl">Zona</td>' + PY.altmans.map(r => {
    const cls = r.zona==='Segura' ? 'ng' : r.zona==='Cinza' ? 'na' : 'nr';
    return `<td class="${cls}">${r.zona}</td>`;
  }).join('');
  tb.appendChild(trZ);
  el.appendChild(tb);
})();

// Score
mk('c_sc1', 'bar', AN, [
  { data: PY.scores.map(s=>s.liq), backgroundColor:'rgba(55,138,221,.65)', borderRadius:3, barPercentage:.2, stack:'s' },
  { data: PY.scores.map(s=>s.end), backgroundColor:'rgba(226,75,74,.65)',  borderRadius:3, barPercentage:.2, stack:'s' },
  { data: PY.scores.map(s=>s.ren), backgroundColor:'rgba(239,159,39,.65)', borderRadius:3, barPercentage:.2, stack:'s' },
  { data: PY.scores.map(s=>s.cai), backgroundColor:'rgba(29,158,117,.65)', borderRadius:3, barPercentage:.2, stack:'s' },
  { data: PY.scores.map(s=>s.sol), backgroundColor:'rgba(155,135,245,.65)',borderRadius:3, barPercentage:.2, stack:'s' },
], { scales:{ x:{stacked:true,grid:{display:false},ticks:{color:'#aaa',font:{size:10}}}, y:{stacked:true,grid:{color:'rgba(0,0,0,.05)'},ticks:{color:'#aaa',font:{size:10},callback:v=>v+'/100'}} }, plugins:{legend:{display:false}} });

(function() {
  const el = document.getElementById('score-bars');
  if (!el) return;
  const dims = [['Liquidez (20%)',s=>s.liq],['Endividamento (25%)',s=>s.end],['Rentab. (20%)',s=>s.ren],['Fluxo caixa (25%)',s=>s.cai],['Solvência (10%)',s=>s.sol]];
  AN.forEach((p,i) => {
    const h = document.createElement('div');
    h.style.cssText = 'font-size:9px;color:#aaa;text-transform:uppercase;letter-spacing:.05em;margin:8px 0 4px;font-weight:600';
    h.textContent = p + ' — Composto: ' + PY.scores[i].comp.toFixed(1) + '/100';
    el.appendChild(h);
    dims.forEach(([lbl,fn]) => {
      const v = fn(PY.scores[i]);
      const color = v > 70 ? '#e24b4a' : v > 50 ? '#ef9f27' : '#1d9e75';
      const row = document.createElement('div');
      row.className = 'sbar';
      row.innerHTML = `<span class="sbar-lbl">${lbl}</span><div class="sbar-bg"><div class="sbar-fill" style="width:${v}%;background:${color}"></div></div><span class="sbar-val" style="color:${color}">${v.toFixed(1)}</span>`;
      el.appendChild(row);
    });
  });
})();

// IQR
mk('c_iq1', 'bar', AN, [
  { data: PY.divida_bruta, backgroundColor: PY.divida_bruta.map(v=>v>PY.iqr_hi_dv?'#e24b4a':'#378add'), borderRadius:4, barPercentage:.6, order:2 },
  { data:[PY.iqr_hi_dv,PY.iqr_hi_dv,PY.iqr_hi_dv,PY.iqr_hi_dv], type:'line', borderColor:'rgba(239,159,39,.8)', borderWidth:1.5, borderDash:[4,3], pointRadius:0, fill:false, order:1 },
], { scales:{ x:{grid:{display:false},ticks:{color:'#aaa',font:{size:10}}}, y:{grid:{color:'rgba(0,0,0,.05)'},ticks:{color:'#aaa',font:{size:10},callback:v=>v+'bi'}} } });

mk('c_iq2', 'bar', AN, [
  { data: PY.cobert_j, backgroundColor: PY.cobert_j.map(v=>v<PY.iqr_lo_cb?'#e24b4a':'#1d9e75'), borderRadius:4, barPercentage:.6, order:2 },
  { data:[PY.iqr_lo_cb,PY.iqr_lo_cb,PY.iqr_lo_cb,PY.iqr_lo_cb], type:'line', borderColor:'rgba(239,159,39,.8)', borderWidth:1.5, borderDash:[4,3], pointRadius:0, fill:false, order:1 },
  { data:[1,1,1,1], type:'line', borderColor:'rgba(226,75,74,.4)', borderWidth:1, borderDash:[3,3], pointRadius:0, fill:false, order:1 },
], { scales:{ x:{grid:{display:false},ticks:{color:'#aaa',font:{size:10}}}, y:{grid:{color:'rgba(0,0,0,.05)'},ticks:{color:'#aaa',font:{size:10},callback:v=>v.toFixed(2)+'×'},min:0,max:2.5} } });

// ── PROJEÇÕES ─────────────────────────────────────────────────────────────────
mk('c_proj1', 'line', ANp, [
  { data:[...PY.divida_bruta,...PY.proj_dv.map(p=>p.yp)], borderColor:'#e24b4a', borderWidth:2, fill:false, tension:0,
    pointRadius:[5,5,5,5,6,6], pointBackgroundColor:[...Array(4).fill('#e24b4a'),...Array(2).fill('#8b0000')],
    segment:{borderDash:ctx=>ctx.p0DataIndex>=3?[5,3]:undefined} },
  { data:[null,null,null,PY.divida_bruta[3],...PY.proj_dv.map(p=>p.hi)], borderColor:'rgba(239,159,39,.4)', borderWidth:1, borderDash:[3,3], pointRadius:0, fill:false },
  { data:[null,null,null,PY.divida_bruta[3],...PY.proj_dv.map(p=>p.lo)], borderColor:'rgba(239,159,39,.4)', borderWidth:1, borderDash:[3,3], pointRadius:0, fill:false },
  { data:[...PY.lucro_liq,...PY.proj_ll.map(p=>p.yp)], borderColor:'#378add', borderWidth:2, fill:false, tension:0,
    pointRadius:[5,5,5,5,6,6], pointBackgroundColor:[...PY.lucro_liq.map(v=>v>=0?'#378add':'#e24b4a'),...Array(2).fill('#8b0000')],
    segment:{borderDash:ctx=>ctx.p0DataIndex>=3?[5,3]:undefined} },
], { scales:{ x:{grid:{display:false},ticks:{color:'#aaa',font:{size:10}}}, y:{grid:{color:'rgba(0,0,0,.05)'},ticks:{color:'#aaa',font:{size:10},callback:v=>v.toFixed(0)+'bi'}} } });

mk('c_cen', 'line', ['2024/25','2025/26p','2026/27p'], [
  { data: PY.cen_base.map(r=>r.cob), borderColor:'#e24b4a', borderWidth:2, fill:false, tension:.3, pointRadius:5, pointBackgroundColor:'#e24b4a' },
  { data: PY.cen_recu.map(r=>r.cob), borderColor:'#1d9e75', borderWidth:2, fill:false, tension:.3, pointRadius:5, pointBackgroundColor:'#1d9e75' },
  { data: PY.cen_coll.map(r=>r.cob), borderColor:'#888', borderWidth:2, fill:false, tension:.3, pointRadius:5, pointBackgroundColor:'#888', borderDash:[5,3] },
  { data:[1,1,1], borderColor:'rgba(226,75,74,.35)', borderWidth:1.5, borderDash:[4,3], pointRadius:0, fill:false },
], { scales:{ x:{grid:{display:false},ticks:{color:'#aaa',font:{size:10}}}, y:{grid:{color:'rgba(0,0,0,.05)'},ticks:{color:'#aaa',font:{size:10},callback:v=>v.toFixed(2)+'×'},min:0,max:1.1} } });

(function() {
  const ep = document.getElementById('tbl_proj');
  if (!ep) return;
  ep.innerHTML = '<thead><tr><th>Indicador</th><th>2025/26p</th><th>IC min</th><th>IC max</th></tr></thead>';
  const tb = document.createElement('tbody');
  const projs = [
    ['Dívida (bi)', PY.ols_dv, true],
    ['Lucro liq. (bi)', PY.ols_ll, false],
    ['FCO (bi)', PY.ols_fco, false],
    ['Cobertura ×', PY.ols_cob, false],
    ['Alavancagem ×', PY.ols_alav, true],
    ['Margem liq. %', PY.ols_ml, false],
  ];
  projs.forEach(([n, r, isDanger]) => {
    const yp  = r.a + r.b * 4;
    const rmse = r.rmse;
    const lo  = yp - 1.96 * rmse;
    const hi  = yp + 1.96 * rmse;
    const cls = isDanger ? (yp > 50 ? 'nr' : 'n') : (yp < 0 ? 'nr' : 'ng');
    const tr = document.createElement('tr');
    tr.innerHTML = `<td class="lbl">${n}</td><td class="${cls}">${yp > 0 ? '+' : ''}${yp.toFixed(2)}</td><td class="n">${lo.toFixed(2)}</td><td class="n">${hi.toFixed(2)}</td>`;
    tb.appendChild(tr);
  });
  ep.appendChild(tb);
  const eo = document.getElementById('tbl_ols');
  if (!eo) return;
  eo.innerHTML = '<thead><tr><th>Indicador</th><th>b/período</th><th>R²</th><th>RMSE</th></tr></thead>';
  const tb2 = document.createElement('tbody');
  projs.forEach(([n, r]) => {
    const tr = document.createElement('tr');
    const bCls = r.b > 0 ? 'ng' : 'nr';
    const r2Cls = r.r2 > 0.9 ? 'ng' : r.r2 > 0.7 ? 'na' : 'nr';
    tr.innerHTML = `<td class="lbl">${n}</td><td class="${bCls}">${r.b > 0 ? '+' : ''}${r.b.toFixed(3)}</td><td class="${r2Cls}">${r.r2.toFixed(4)}</td><td class="n">${r.rmse.toFixed(3)}</td>`;
    tb2.appendChild(tr);
  });
  eo.appendChild(tb2);
})();

// ── MACRO ─────────────────────────────────────────────────────────────────────
const elMacro = document.getElementById('c_macro');
if (elMacro) {
  new Chart(elMacro, {
    type:'bar',
    data:{ labels: PY.safras, datasets:[
      { data: PY.prod_cana,  backgroundColor:'rgba(29,158,117,.6)', borderRadius:3, barPercentage:.7, stack:'s' },
      { data: PY.prod_milho, backgroundColor:'rgba(239,159,39,.7)', borderRadius:3, barPercentage:.7, stack:'s' },
      { data: PY.share_milho, type:'line', borderColor:'#9b87f5', borderWidth:2.5, fill:false, tension:.4, pointRadius:4, pointBackgroundColor:'#9b87f5', yAxisID:'y2' },
    ]},
    options:{ responsive:true, maintainAspectRatio:false, plugins:{legend:{display:false}},
      scales:{ x:{stacked:true,grid:{display:false},ticks:{color:'#aaa',font:{size:10}}},
               y:{stacked:true,grid:{color:'rgba(0,0,0,.05)'},ticks:{color:'#aaa',font:{size:10},callback:v=>v+'bi'}},
               y2:{position:'right',grid:{display:false},ticks:{color:'#9b87f5',font:{size:10},callback:v=>v+'%'},min:0,max:40} } }
  });
}

const elCustos = document.getElementById('c_custos');
if (elCustos) {
  new Chart(elCustos, {
    type:'line',
    data:{ labels: PY.safras, datasets:[
      { data: PY.custo_milho, borderColor:'#ef9f27', borderWidth:2.5, fill:false, tension:.4, pointRadius:4, pointBackgroundColor:'#ef9f27' },
      { data: PY.custo_cana,  borderColor:'#e24b4a', borderWidth:2.5, fill:false, tension:.4, pointRadius:4, pointBackgroundColor:'#e24b4a' },
      { data: PY.custo_cana,  borderColor:'transparent', backgroundColor:'rgba(155,135,245,.07)', fill:'-1', tension:.4, pointRadius:0 },
    ]},
    options:{ responsive:true, maintainAspectRatio:false, plugins:{legend:{display:false}},
      scales:{ x:{grid:{display:false},ticks:{color:'#aaa',font:{size:10}}},
               y:{grid:{color:'rgba(0,0,0,.05)'},ticks:{color:'#aaa',font:{size:10},callback:v=>'R$'+v.toFixed(2)},min:1.3,max:2.6} } }
  });
}

(function() {
  const el = document.getElementById('cagr-bars');
  if (!el) return;
  [['Milho (produção)',31,'#ef9f27'],['Cana (produção)',0.5,'#1d9e75'],['Inpasa (receita)',50,'#1d9e75'],['Raízen (receita)',5,'#e24b4a'],['Gap custo (+860%)',100,'#e24b4a']].forEach(([lbl,v,c]) => {
    const pct = v;
    const row = document.createElement('div');
    row.className='sbar';
    row.innerHTML=`<span class="sbar-lbl">${lbl}</span><div class="sbar-bg"><div class="sbar-fill" style="width:${Math.min(pct,100)}%;background:${c}"></div></div><span class="sbar-val" style="color:${c}">${v>=100?'+'+v+'%':'+'+v+'%/a'}</span>`;
    el.appendChild(row);
  });
})();

// ── CONCORRENTES ──────────────────────────────────────────────────────────────
mk('c_cmp','bar',['Margem liq. %','Alavancagem ×'],[
  { data:[-8.8,5.3], backgroundColor:'#e24b4a', borderRadius:4, barPercentage:.25 },
  { data:[18.4,1.7], backgroundColor:'#1d9e75', borderRadius:4, barPercentage:.25 },
  { data:[8.8,null], backgroundColor:'#378add', borderRadius:4, barPercentage:.25 },
],{ scales:{ x:{grid:{display:false},ticks:{color:'#aaa',font:{size:10}}}, y:{grid:{color:'rgba(0,0,0,.05)'},ticks:{color:'#aaa',font:{size:10}}} } });

mk('c_inp','bar',PY.safras,[
  { data:[0.376,0.8,2.1,5.4,9.8,13.6], backgroundColor:'rgba(29,158,117,.7)', borderRadius:4, barPercentage:.6 }
],{ scales:{ x:{grid:{display:false},ticks:{color:'#aaa',font:{size:10}}}, y:{grid:{color:'rgba(0,0,0,.05)'},ticks:{color:'#aaa',font:{size:10},callback:v=>v+'bi'}} } });

// ── RED FLAGS ─────────────────────────────────────────────────────────────────
(function() {
  const el = document.getElementById('flags-fin');
  if (!el) return;
  const flagsData = [
    {p:'2021/22',n:'ALERTA',c:'DFC',t:'FCL negativo desde o IPO',d:'Capex supera caixa gerado em todos os anos',v:'−R$ 3,6 bi',vc:'a'},
    {p:'2021/22',n:'ALERTA',c:'DFC',t:'Capex/FCO acima de 1,5×',d:'Investimento estruturalmente insustentável',v:'1,50×',vc:'a'},
    {p:'2021/22',n:'ALERTA',c:'BALANÇO',t:'Liquidez corrente abaixo de 1,2×',d:'Margem de segurança estreita desde o 1º ano',v:'1,21×',vc:'a'},
    {p:'2022/23',n:'ALERTA',c:'BALANÇO',t:'Alavancagem ultrapassa 3,0×',d:'Dívida dobra de R$ 18 bi para R$ 37 bi em 12 meses',v:'3,4×',vc:'a'},
    {p:'2022/23',n:'ALERTA',c:'DFC',t:'Cobertura de juros abaixo de 1,5×',d:'Margem de cobertura deteriorando rapidamente',v:'1,48×',vc:'a'},
    {p:'2023/24',n:'CRÍTICO',c:'DRE',t:'1º prejuízo líquido da história',d:'Margem negativa pela 1ª vez desde o IPO (2011)',v:'−R$ 3,1 bi',vc:'r'},
    {p:'2023/24',n:'CRÍTICO',c:'DRE',t:'EBIT negativo — operação no vermelho',d:'Margem EBIT: −1,4% — custo operacional supera receita',v:'−1,4%',vc:'r'},
    {p:'2023/24',n:'CRÍTICO',c:'DFC',t:'FCO não cobre mais os juros pagos',d:'Cobertura 0,70× — caixa abaixo do mínimo crítico',v:'0,70×',vc:'r'},
    {p:'2023/24',n:'CRÍTICO',c:'BALANÇO',t:'Liquidez corrente abaixo de 1,0×',d:'Passivo circulante supera ativo circulante',v:'0,91×',vc:'r'},
    {p:'2024/25',n:'CRÍTICO',c:'DRE',t:'Prejuízo de R$ 17,2 bi',d:'Margem líquida −8,8% · inclui impairment E2G R$ 11,1 bi',v:'−8,8%',vc:'r'},
    {p:'2024/25',n:'CRÍTICO',c:'DFC',t:'Cobertura 0,48× — colapso de caixa',d:'FCO (R$ 5,8 bi) cobre menos da metade dos juros (R$ 12,1 bi)',v:'0,48×',vc:'r'},
    {p:'2024/25',n:'CRÍTICO',c:'BALANÇO',t:'Patrimônio líquido NEGATIVO',d:'PL = −R$ 2,6 bi — tecnicamente insolvente pelo balanço',v:'−R$ 2,6 bi',vc:'r'},
    {p:'2024/25',n:'CRÍTICO',c:'BALANÇO',t:'Endividamento geral: 88%',d:'Acima do limiar crítico de 80% — quase todo ativo é dívida',v:'88%',vc:'r'},
  ];
  flagsData.forEach(f => {
    const nc = f.n === 'CRÍTICO' ? 'c' : 'al';
    const div = document.createElement('div');
    div.className = 'fl';
    div.innerHTML = `<span class="fp ${nc}">${f.n}</span><span class="fc">${f.c}</span><div class="ft"><div class="ft-t">${f.p} — ${f.t}</div><div class="ft-d">${f.d}</div></div><div class="fv ${f.vc}">${f.v}</div>`;
    el.appendChild(div);
  });
})();

// ── CRONOLOGIA — CONVERGÊNCIA ─────────────────────────────────────────────────
const convScores = PY.scores.map(s => s.comp);
const convAlt    = PY.altmans.map(r => r.Z);
const elConv = document.getElementById('c_conv');
if (elConv) {
  new Chart(elConv, {
    type:'line',
    data:{ labels: AN, datasets:[
      { data: convScores, borderColor:'#e24b4a', borderWidth:2.5, fill:true, backgroundColor:'rgba(226,75,74,.07)', tension:.4,
        pointRadius:5, pointBackgroundColor: convScores.map(s=>s>70?'#e24b4a':s>50?'#ef9f27':'#1d9e75'), yAxisID:'y' },
      { data:[50,50,50,50], borderColor:'rgba(239,159,39,.4)', borderWidth:1.5, borderDash:[4,3], pointRadius:0, fill:false, yAxisID:'y' },
      { data: convAlt, borderColor:'#9b87f5', borderWidth:2, fill:false, tension:.4,
        pointRadius:5, pointBackgroundColor: convAlt.map(z=>z>2.99?'#1d9e75':z>=1.81?'#ef9f27':'#e24b4a'), yAxisID:'y2' },
      { data:[1.81,1.81,1.81,1.81], borderColor:'rgba(155,135,245,.3)', borderWidth:1, borderDash:[4,3], pointRadius:0, fill:false, yAxisID:'y2' },
    ]},
    options:{ responsive:true, maintainAspectRatio:false, plugins:{legend:{display:false}},
      scales:{
        x:  { grid:{display:false}, ticks:{color:'#aaa',font:{size:10}} },
        y:  { grid:{color:'rgba(0,0,0,.05)'}, ticks:{color:'#aaa',font:{size:10},callback:v=>v+'/100'}, min:0, max:100 },
        y2: { position:'right', grid:{display:false}, ticks:{color:'#9b87f5',font:{size:10},callback:v=>v.toFixed(1)+'Z'}, min:-0.5, max:3.5 }
      } }
  });
}

// ── RESUMO ────────────────────────────────────────────────────────────────────
(function() {
  const el = document.getElementById('tbl_est');
  if (!el) return;
  [
    ['Var. lucro 2021→2025', '−919%', 'nr'],
    ['Var. dívida 2021→2025', '+280%', 'nr'],
    ['FCL acumulado 4 anos', '−R$ 24,2 bi', 'nr'],
    ['R² OLS dívida', PY.ols_dv.r2.toFixed(4), 'ng'],
    ['Altman Z (2024/25)', PY.altmans[3].Z.toFixed(3), 'nr'],
    ['Score risco (2024/25)', PY.scores[3].comp.toFixed(1)+'/100', 'nr'],
    ['Red flags detectados', '38', 'nr'],
    ['RE total (plano)', 'R$ 98,6 bi', 'nr'],
  ].forEach(([n,v,cls]) => {
    const tr = document.createElement('tr');
    tr.innerHTML = `<td class="lbl">${n}</td><td class="${cls}">${v}</td>`;
    el.appendChild(tr);
  });
})();

// ── NAVEGAÇÃO ─────────────────────────────────────────────────────────────────
function nav(id, btn) {
  document.querySelectorAll('.sec').forEach(s => s.classList.remove('on'));
  document.querySelectorAll('.nav-btn').forEach(b => b.classList.remove('on'));
  document.getElementById('s-' + id).classList.add('on');
  btn.classList.add('on');
  window.scrollTo(0, 0);
}
function stab(group, id, btn) {
  document.querySelectorAll('[id^="'+group+'-"]').forEach(el => el.classList.remove('on'));
  document.querySelectorAll('.stab').forEach(t => t.classList.remove('on'));
  document.getElementById(group+'-'+id).classList.add('on');
  btn.classList.add('on');
}
</script>
</body>
</html>"""

# ══════════════════════════════════════════════════════════════════════════════
# 5. INJETAR JSON E SALVAR
# ══════════════════════════════════════════════════════════════════════════════

html_final = HTML.replace('__DADOS_JSON__', dados_json)

output_file = 'raizen_dashboard_completo.html'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(html_final)

tamanho = os.path.getsize(output_file)
print(f"\nArquivo gerado: {output_file}")
print(f"Tamanho: {tamanho/1024:.0f} KB")
print("\nAbrindo no navegador...")
webbrowser.open(f'file://{os.path.abspath(output_file)}')

Dados calculados em Python puro:
  Altman Z 2024/25 = 1.9011 (Cinza)
  Score risco 2024/25 = 46.3/100
  OLS dívida: b=16.970 bi/período, R²=0.9983
  Proj. dívida 2025/26 = R$ 86.8 bi [IC: 85.2–88.4]
  Z-Score LL 2024/25 = -1.450σ
  Limite IQR superior dívida = R$ 92.6 bi

Arquivo gerado: raizen_dashboard_completo.html
Tamanho: 71 KB

Abrindo no navegador...


True